In [1]:
!pip install -q parsbench
!pip install -q -U transformers accelerate bitsandbytes sentencepiece

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.0/254.0 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.2/887.2 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 15.7 MB/s eta 0:00:00


In [3]:
from huggingface_hub import login

login()

In [4]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_NAME = "PartAI/Dorna2-Llama3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16
)

model.eval()

print("Loaded:", MODEL_NAME)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.3k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Loaded: PartAI/Dorna2-Llama3.1-8B-Instruct


In [21]:
from datasets import load_dataset
import pandas as pd

BASE_URL = (
    "https://raw.githubusercontent.com/persiannlp/parsinlu/"
    "master/data/multiple-choice/"
)

FILES = {
    "math_and_logic": BASE_URL + "test_ml.jsonl",
    "common_knowledge": BASE_URL + "test_ck.jsonl",
    "literature": BASE_URL + "test_lit.jsonl",
}

frames = []

for category, url in FILES.items():
    ds = load_dataset(
        "json",
        data_files=url,
        split="train"
    )

    df = ds.to_pandas()

    # Make sure our category label is consistent
    df["category"] = category

    frames.append(df)

eval_df = pd.concat(
    frames,
    ignore_index=True
)

eval_df["example_id"] = range(1, len(eval_df) + 1)

print("Total test examples:", len(eval_df))
print()
print(eval_df["category"].value_counts())
print()
print(eval_df.columns.tolist())

display(eval_df.head())

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Total test examples: 1050

category
math_and_logic      350
common_knowledge    350
literature          350
Name: count, dtype: int64

['answer', 'candidates', 'category', 'question', 'id', 'example_id']


,answer,candidates,category,question,id,example_id
0,2,"[2A, 2A+B, 3A+B, A-B]",math_and_logic,تفاوت سن علیرضا و خواهرش A سال است B سال دیگر ...,Alefba-976660247951-77_Omoomi_Sample_Hoosh5__e...,1
1,2,"[۴۶, ۴۱, ۵۱, ۳۶]",math_and_logic,در ادامه این رشته چه عددی باید نوشت؟ ۹۱،۸۶،۷۶،...,Alefba-661560247951-31_Omoomi_Sample_Hoosh2__e...,2
2,3,"[10000, 100, 1000, 500]",math_and_logic,50 تا 20 تا برابر است با ......,http://dl.biamoz.com/5/riazi/nobat2/نمونه-سوال...,3
3,1,"[17, 11, 14, 15]",math_and_logic,در ادامه این رشته چه عددی باید نوشت؟ ...,Alefba-976660247951-77_Omoomi_Sample_Hoosh5__e...,4
4,1,"[۴, ۲, ۳, ۱]",math_and_logic,مساحت مربع ۸ ،p برابر مساحت مربع Q است. نسبت ق...,Alefba-661560247951-31_Omoomi_Sample_Hoosh2__e...,5


In [23]:
print(eval_df.columns.tolist())
display(eval_df.head(3))

['answer', 'candidates', 'category', 'question', 'id', 'example_id']


,answer,candidates,category,question,id,example_id
0,2,"[2A, 2A+B, 3A+B, A-B]",math_and_logic,تفاوت سن علیرضا و خواهرش A سال است B سال دیگر ...,Alefba-976660247951-77_Omoomi_Sample_Hoosh5__e...,1
1,2,"[۴۶, ۴۱, ۵۱, ۳۶]",math_and_logic,در ادامه این رشته چه عددی باید نوشت؟ ۹۱،۸۶،۷۶،...,Alefba-661560247951-31_Omoomi_Sample_Hoosh2__e...,2
2,3,"[10000, 100, 1000, 500]",math_and_logic,50 تا 20 تا برابر است با ......,http://dl.biamoz.com/5/riazi/nobat2/نمونه-سوال...,3


In [27]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
import json
import os
import re
import time
import torch


BASE_DIR = Path(
    "/content/drive/MyDrive/llm_benchmark/parsbench"
)

BASE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_SLUG = "dorna2-llama3.1-8b"

CHECKPOINT_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_mcq_test_checkpoint.csv"
)

SUMMARY_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_mcq_test_summary.csv"
)

print("Checkpoint:", CHECKPOINT_PATH)

Checkpoint: /content/drive/MyDrive/llm_benchmark/parsbench/dorna2-llama3.1-8b_parsinlu_mcq_test_checkpoint.csv


In [28]:
def build_mcq_prompt(row):
    question = str(row["question"]).strip()

    candidates = row["candidates"]

    # Handle numpy arrays / lists
    if isinstance(candidates, np.ndarray):
        candidates = candidates.tolist()

    # Handle string representation if needed
    if isinstance(candidates, str):
        try:
            parsed = json.loads(candidates)
            if isinstance(parsed, list):
                candidates = parsed
        except Exception:
            pass

    if not isinstance(candidates, (list, tuple)):
        raise ValueError(
            f"Unexpected candidates format: {type(candidates)}"
        )

    options = "\n".join(
        f"{i}. {choice}"
        for i, choice in enumerate(candidates, start=1)
    )

    prompt = f"""در ادامه، به شما یک سوال چند گزینه‌ای به زبان فارسی نشان داده می شود. شما باید بر اساس دانش خود به سوال پاسخ دهید. پاسخ خود را از بین گزینه‌های داده شده انتخاب کنید.
فقط عدد متناظر با گزینه درست را خروجی بده.

سوال:
'''{question}'''
گزینه ها:
'''{options}'''
جواب:"""

    return prompt

In [29]:
print(build_mcq_prompt(eval_df.iloc[0]))
print("TARGET:", eval_df.iloc[0]["answer"])

در ادامه، به شما یک سوال چند گزینه‌ای به زبان فارسی نشان داده می شود. شما باید بر اساس دانش خود به سوال پاسخ دهید. پاسخ خود را از بین گزینه‌های داده شده انتخاب کنید.
فقط عدد متناظر با گزینه درست را خروجی بده.

سوال:
'''تفاوت سن علیرضا و خواهرش A سال است B سال دیگر سن علیرضا دوبرابر سن امروز خواهرش خواهد بود .سن خواهر علیرضا کدام است؟'''
گزینه ها:
'''1. 2A
2. 2A+B
3. 3A+B
4. A-B'''
جواب:
TARGET: 2


In [30]:
@torch.inference_mode()
def generate_mcq_answer(prompt):
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    chat_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        chat_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    torch.cuda.synchronize()
    start = time.perf_counter()

    outputs = model.generate(
        **inputs,
        max_new_tokens=32,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    torch.cuda.synchronize()
    latency = time.perf_counter() - start

    generated_ids = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    completion = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    output_tokens = len(generated_ids)

    return completion, latency, output_tokens

In [39]:
import re
import unicodedata

DIGIT_TRANSLATION = str.maketrans({
    # Persian
    "۰": "0",
    "۱": "1",
    "۲": "2",
    "۳": "3",
    "۴": "4",
    "۵": "5",
    "۶": "6",
    "۷": "7",
    "۸": "8",
    "۹": "9",

    # Arabic
    "٠": "0",
    "١": "1",
    "٢": "2",
    "٣": "3",
    "٤": "4",
    "٥": "5",
    "٦": "6",
    "٧": "7",
    "٨": "8",
    "٩": "9",
})


def canonicalize_text(text):
    if text is None:
        return ""

    text = str(text)

    # Unicode normalization
    text = unicodedata.normalize("NFKC", text)

    # Persian/Arabic digits -> ASCII
    text = text.translate(DIGIT_TRANSLATION)

    # Arabic/Persian character variants
    text = (
        text
        .replace("ي", "ی")
        .replace("ك", "ک")
    )

    # Remove markdown
    text = text.replace("**", "")

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def normalize_mcq_answer(text, candidates=None):
    if text is None:
        return "-1"

    text = canonicalize_text(text)

    # ---------------------------
    # 1. Exact numeric response
    # ---------------------------
    if text in {"1", "2", "3", "4"}:
        return text

    # ---------------------------
    # 2. Explicit option number
    # ---------------------------
    patterns = [
        r"گزینه\s*(?:صحیح|درست)?\s*[:：\-]?\s*([1-4])",
        r"جواب\s*(?:صحیح|درست)?\s*[:：\-]?\s*([1-4])",
        r"پاسخ\s*(?:صحیح|درست)?\s*[:：\-]?\s*([1-4])",
    ]

    for pattern in patterns:
        match = re.search(pattern, text)

        if match:
            return match.group(1)

    # ---------------------------
    # 3. Response begins with number
    # ---------------------------
    match = re.match(
        r"^\s*([1-4])(?:\s*[\.\-\):،]|$)",
        text
    )

    if match:
        return match.group(1)

    # ---------------------------
    # 4. Model returned option TEXT
    # ---------------------------
    if candidates is not None:

        # Remove common response prefixes
        cleaned = re.sub(
            r"^(?:گزینه\s*(?:صحیح|درست)?|"
            r"جواب\s*(?:صحیح|درست)?|"
            r"پاسخ\s*(?:صحیح|درست)?)"
            r"\s*[:：\-]?\s*",
            "",
            text
        )

        cleaned = canonicalize_text(cleaned)

        for i, candidate in enumerate(
            candidates,
            start=1
        ):
            candidate_clean = canonicalize_text(
                candidate
            )

            if cleaned == candidate_clean:
                return str(i)

    return "-1"

In [40]:
tests = [
    "2",
    "۲",
    "گزینه 4: A-B",
    "جواب صحیح: **2**",
    "جواب: 4. باران است با برف",
    "گزینه صحیح: **2. حافظ، عرفانی، غزلیات**",
]

for x in tests:
    print(
        repr(x),
        "->",
        normalize_mcq_answer(x)
    )

'2' -> 2
'۲' -> 2
'گزینه 4: A-B' -> 4
'جواب صحیح: **2**' -> 2
'جواب: 4. باران است با برف' -> 4
'گزینه صحیح: **2. حافظ، عرفانی، غزلیات**' -> 2


In [33]:
def load_checkpoint(path):
    if not path.exists():
        print("No checkpoint found.")
        return {}

    df = pd.read_csv(path)

    results = {}

    for _, row in df.iterrows():
        results[int(row["example_id"])] = row.to_dict()

    print(
        f"Loaded checkpoint: {len(results)} rows"
    )

    return results


def save_checkpoint(results, path):
    if not results:
        return

    df = pd.DataFrame(
        list(results.values())
    ).sort_values("example_id")

    tmp_path = str(path) + ".tmp"

    df.to_csv(
        tmp_path,
        index=False,
        encoding="utf-8-sig"
    )

    os.replace(
        tmp_path,
        path
    )

In [42]:
def run_mcq_evaluation(
    dataset,
    checkpoint_path,
    limit=None
):
    results = load_checkpoint(
        checkpoint_path
    )

    successful_ids = {
        example_id
        for example_id, result in results.items()
        if result.get("status") == "success"
    }

    if limit is not None:
        work_df = dataset.iloc[:limit].copy()
    else:
        work_df = dataset.copy()

    total = len(work_df)

    already_done = sum(
        int(row["example_id"]) in successful_ids
        for _, row in work_df.iterrows()
    )

    print(f"Total examples: {total}")
    print(f"Already completed: {already_done}")
    print(f"Remaining: {total - already_done}")
    print("-" * 60)

    for position, (_, row) in enumerate(
        work_df.iterrows(),
        start=1
    ):
        example_id = int(
            row["example_id"]
        )

        if example_id in successful_ids:
            continue

        prompt = build_mcq_prompt(row)

        target = str(
            row["answer"]
        ).translate(
            DIGIT_TRANSLATION
        ).strip()

        category = str(
            row["category"]
        )

        try:
            torch.cuda.reset_peak_memory_stats()

            completion, latency, output_tokens = (
                generate_mcq_answer(prompt)
            )

            normalized = normalize_mcq_answer(
                completion,
                row["candidates"]
            )

            correct = int(
                normalized == target
            )

            peak_vram_gb = (
                torch.cuda.max_memory_allocated()
                / (1024 ** 3)
            )

            tokens_per_sec = (
                output_tokens / latency
                if latency > 0
                else None
            )

            results[example_id] = {
                "example_id": example_id,
                "category": category,
                "question": row["question"],
                "candidates": json.dumps(
                    list(row["candidates"]),
                    ensure_ascii=False
                ),
                "target": target,
                "raw_completion": completion,
                "normalized_completion": normalized,
                "correct": correct,
                "latency_sec": latency,
                "output_tokens": output_tokens,
                "tokens_per_sec": tokens_per_sec,
                "peak_vram_gb": peak_vram_gb,
                "status": "success",
                "error": "",
            }

            successful_ids.add(
                example_id
            )

        except Exception as e:

            results[example_id] = {
                "example_id": example_id,
                "category": category,
                "question": row["question"],
                "candidates": json.dumps(
                    list(row["candidates"]),
                    ensure_ascii=False
                ),
                "target": target,
                "raw_completion": "",
                "normalized_completion": "-1",
                "correct": 0,
                "latency_sec": None,
                "output_tokens": None,
                "tokens_per_sec": None,
                "peak_vram_gb": None,
                "status": "error",
                "error": str(e),
            }

            print(
                f"\nERROR example {example_id}: {e}"
            )

        # Save after EVERY example
        save_checkpoint(
            results,
            checkpoint_path
        )

        completed = len(
            successful_ids.intersection(
                set(
                    work_df[
                        "example_id"
                    ].astype(int)
                )
            )
        )

        print(
            f"\r"
            f"{completed}/{total} completed | "
            f"ID {example_id} | "
            f"{category} | "
            f"target={target} | "
            f"pred={normalized}",
            end=""
        )

    print("\nEvaluation finished.")

    return pd.DataFrame(
        list(results.values())
    ).sort_values(
        "example_id"
    )

In [43]:
results = load_checkpoint(
    CHECKPOINT_PATH
)

for example_id, result in results.items():

    if result.get("status") != "success":
        continue

    # Find corresponding benchmark row
    row = eval_df[
        eval_df["example_id"] == example_id
    ].iloc[0]

    normalized = normalize_mcq_answer(
        result["raw_completion"],
        row["candidates"]
    )

    target = str(
        row["answer"]
    ).translate(
        DIGIT_TRANSLATION
    ).strip()

    result["normalized_completion"] = normalized
    result["correct"] = int(
        normalized == target
    )

save_checkpoint(
    results,
    CHECKPOINT_PATH
)

print("Checkpoint rescored.")

Loaded checkpoint: 30 rows
Checkpoint rescored.


In [44]:
rescored_df = pd.DataFrame(
    results.values()
).sort_values("example_id")

first_30 = rescored_df[
    rescored_df["example_id"].between(
        1, 30
    )
]

display(
    first_30[
        [
            "example_id",
            "category",
            "target",
            "raw_completion",
            "normalized_completion",
            "correct",
        ]
    ]
)

print(
    "Accuracy:",
    first_30["correct"].mean()
)

,example_id,category,target,raw_completion,normalized_completion,correct
0,1,math_and_logic,2,گزینه 2: 2A+B,2,1
1,2,math_and_logic,2,گزینه 2: ۴۱,2,1
2,3,math_and_logic,3,گزینه 2,2,0
3,4,math_and_logic,1,گزینه 2: 11,2,0
4,5,math_and_logic,1,گزینه 2,2,0
5,6,math_and_logic,4,جواب: 4,4,1
6,7,math_and_logic,2,جواب: ۲,2,1
7,8,math_and_logic,2,جواب: ۱,1,0
8,9,math_and_logic,3,گزینه صحیح: ۳. ۳۳.۳۳,3,1
9,10,math_and_logic,1,جواب: 2,2,0


Accuracy: 0.4


In [45]:
smoke_eval_df = (
    eval_df
    .groupby(
        "category",
        group_keys=False
    )
    .head(10)
    .copy()
)

print(
    smoke_eval_df["category"]
    .value_counts()
)

category
math_and_logic      10
common_knowledge    10
literature          10
Name: count, dtype: int64


In [46]:
smoke_results_df = run_mcq_evaluation(
    smoke_eval_df,
    CHECKPOINT_PATH
)

Loaded checkpoint: 30 rows
Total examples: 30
Already completed: 10
Remaining: 20
------------------------------------------------------------
30/30 completed | ID 710 | literature | target=4 | pred=4
Evaluation finished.


In [47]:
smoke_ids = set(
    smoke_eval_df["example_id"]
)

balanced_smoke = smoke_results_df[
    smoke_results_df[
        "example_id"
    ].isin(smoke_ids)
].copy()

summary = (
    balanced_smoke
    .groupby("category")
    .agg(
        questions=("example_id", "count"),
        correct=("correct", "sum"),
        accuracy=("correct", "mean"),
        avg_latency_sec=(
            "latency_sec",
            "mean"
        ),
        avg_tokens_per_sec=(
            "tokens_per_sec",
            "mean"
        ),
        peak_vram_gb=(
            "peak_vram_gb",
            "max"
        ),
    )
)

display(summary)

overall_accuracy = (
    balanced_smoke["correct"].mean()
)

print(
    f"Overall balanced smoke accuracy: "
    f"{overall_accuracy:.2%}"
)

,questions,correct,accuracy,avg_latency_sec,avg_tokens_per_sec,peak_vram_gb
category,,,,,,
common_knowledge,10,3,0.3,0.906120,7.581787,5.471202
literature,10,4,0.4,0.982208,7.587576,5.491307
math_and_logic,10,5,0.5,0.835855,8.969819,5.471202


Overall balanced smoke accuracy: 40.00%


In [48]:
results_df = run_mcq_evaluation(
    eval_df,
    CHECKPOINT_PATH
)

Loaded checkpoint: 50 rows
Total examples: 1050
Already completed: 50
Remaining: 1000
------------------------------------------------------------
1050/1050 completed | ID 1050 | literature | target=1 | pred=2
Evaluation finished.


In [50]:
successful = results_df[
    results_df["status"] == "success"
].copy()

print("Successful rows:", len(successful))
print("Errors:", (results_df["status"] == "error").sum())

Successful rows: 1050
Errors: 0


In [51]:
category_summary = (
    successful
    .groupby("category")
    .agg(
        questions=("example_id", "count"),
        correct=("correct", "sum"),
        accuracy=("correct", "mean"),
        avg_latency_sec=("latency_sec", "mean"),
        avg_tokens_per_sec=("tokens_per_sec", "mean"),
        peak_vram_gb=("peak_vram_gb", "max"),
    )
    .reset_index()
)

display(category_summary)

,category,questions,correct,accuracy,avg_latency_sec,avg_tokens_per_sec,peak_vram_gb
0,common_knowledge,350,143,0.408571,0.697720,9.719453,5.484757
1,literature,350,131,0.374286,0.695774,9.567498,5.509859
2,math_and_logic,350,114,0.325714,0.779778,10.257342,5.487520


In [52]:
overall_summary = pd.DataFrame([{
    "category": "overall",
    "questions": len(successful),
    "correct": int(successful["correct"].sum()),
    "accuracy": successful["correct"].mean(),
    "avg_latency_sec": successful["latency_sec"].mean(),
    "avg_tokens_per_sec": successful["tokens_per_sec"].mean(),
    "peak_vram_gb": successful["peak_vram_gb"].max(),
}])

In [53]:
final_summary = pd.concat(
    [
        category_summary,
        overall_summary
    ],
    ignore_index=True
)

final_summary.insert(
    0,
    "model_name",
    MODEL_NAME
)

final_summary.insert(
    1,
    "model_slug",
    MODEL_SLUG
)

display(final_summary)

,model_name,model_slug,category,questions,correct,accuracy,avg_latency_sec,avg_tokens_per_sec,peak_vram_gb
0,PartAI/Dorna2-Llama3.1-8B-Instruct,dorna2-llama3.1-8b,common_knowledge,350,143,0.408571,0.697720,9.719453,5.484757
1,PartAI/Dorna2-Llama3.1-8B-Instruct,dorna2-llama3.1-8b,literature,350,131,0.374286,0.695774,9.567498,5.509859
2,PartAI/Dorna2-Llama3.1-8B-Instruct,dorna2-llama3.1-8b,math_and_logic,350,114,0.325714,0.779778,10.257342,5.487520
3,PartAI/Dorna2-Llama3.1-8B-Instruct,dorna2-llama3.1-8b,overall,1050,388,0.369524,0.724424,9.848098,5.509859


In [54]:
SUMMARY_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_mcq_test_summary.csv"
)

final_summary.to_csv(
    SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Saved summary to:")
print(SUMMARY_PATH)

Saved summary to:
/content/drive/MyDrive/llm_benchmark/parsbench/dorna2-llama3.1-8b_parsinlu_mcq_test_summary.csv


Gemma 2 2B

In [59]:
MODEL_NAME = MODELS["gemma2-2b-it"]
MODEL_SLUG = "gemma2-2b-it"

load_model(MODEL_NAME)

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Loaded: google/gemma-2-2b-it
GPU: Tesla T4


In [60]:
CHECKPOINT_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_mcq_test_checkpoint.csv"
)

SUMMARY_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_mcq_test_summary.csv"
)

print(CHECKPOINT_PATH)

/content/drive/MyDrive/llm_benchmark/parsbench/gemma2-2b-it_parsinlu_mcq_test_checkpoint.csv


In [61]:
smoke_eval_df = (
    eval_df
    .groupby(
        "category",
        group_keys=False
    )
    .head(10)
    .copy()
)

smoke_results_df = run_mcq_evaluation(
    smoke_eval_df,
    CHECKPOINT_PATH
)

No checkpoint found.
Total examples: 30
Already completed: 0
Remaining: 30
------------------------------------------------------------
30/30 completed | ID 710 | literature | target=4 | pred=4
Evaluation finished.


In [62]:
smoke_ids = set(
    smoke_eval_df["example_id"]
)

balanced_smoke = smoke_results_df[
    smoke_results_df["example_id"].isin(
        smoke_ids
    )
].copy()

summary = (
    balanced_smoke
    .groupby("category")
    .agg(
        questions=("example_id", "count"),
        correct=("correct", "sum"),
        accuracy=("correct", "mean"),
        avg_latency_sec=("latency_sec", "mean"),
        avg_tokens_per_sec=("tokens_per_sec", "mean"),
        peak_vram_gb=("peak_vram_gb", "max"),
    )
)

display(summary)

print(
    f"Overall balanced smoke accuracy: "
    f"{balanced_smoke['correct'].mean():.2%}"
)

,questions,correct,accuracy,avg_latency_sec,avg_tokens_per_sec,peak_vram_gb
category,,,,,,
common_knowledge,10,1,0.1,0.449053,10.971403,7.472050
literature,10,3,0.3,0.529500,12.886326,7.490085
math_and_logic,10,1,0.1,1.120511,14.241063,7.473112


Overall balanced smoke accuracy: 16.67%


In [63]:
results_df = run_mcq_evaluation(
    eval_df,
    CHECKPOINT_PATH
)

Loaded checkpoint: 30 rows
Total examples: 1050
Already completed: 30
Remaining: 1020
------------------------------------------------------------
1050/1050 completed | ID 1050 | literature | target=1 | pred=-1
Evaluation finished.


In [64]:
successful = results_df[
    results_df["status"] == "success"
].copy()

print("Successful rows:", len(successful))
print("Errors:", (results_df["status"] == "error").sum())

Successful rows: 1050
Errors: 0


In [65]:
category_summary = (
    successful
    .groupby("category")
    .agg(
        questions=("example_id", "count"),
        correct=("correct", "sum"),
        accuracy=("correct", "mean"),
        avg_latency_sec=("latency_sec", "mean"),
        avg_tokens_per_sec=("tokens_per_sec", "mean"),
        peak_vram_gb=("peak_vram_gb", "max"),
    )
    .reset_index()
)

display(category_summary)

,category,questions,correct,accuracy,avg_latency_sec,avg_tokens_per_sec,peak_vram_gb
0,common_knowledge,350,120,0.342857,0.356323,14.034180,7.482830
1,literature,350,89,0.254286,0.642326,14.329384,7.501582
2,math_and_logic,350,61,0.174286,1.101844,15.151293,7.482712


In [66]:
overall_summary = pd.DataFrame([{
    "category": "overall",
    "questions": len(successful),
    "correct": int(successful["correct"].sum()),
    "accuracy": successful["correct"].mean(),
    "avg_latency_sec": successful["latency_sec"].mean(),
    "avg_tokens_per_sec": successful["tokens_per_sec"].mean(),
    "peak_vram_gb": successful["peak_vram_gb"].max(),
}])

In [67]:
final_summary = pd.concat(
    [
        category_summary,
        overall_summary
    ],
    ignore_index=True
)

final_summary.insert(
    0,
    "model_name",
    MODEL_NAME
)

final_summary.insert(
    1,
    "model_slug",
    MODEL_SLUG
)

display(final_summary)

,model_name,model_slug,category,questions,correct,accuracy,avg_latency_sec,avg_tokens_per_sec,peak_vram_gb
0,google/gemma-2-2b-it,gemma2-2b-it,common_knowledge,350,120,0.342857,0.356323,14.034180,7.482830
1,google/gemma-2-2b-it,gemma2-2b-it,literature,350,89,0.254286,0.642326,14.329384,7.501582
2,google/gemma-2-2b-it,gemma2-2b-it,math_and_logic,350,61,0.174286,1.101844,15.151293,7.482712
3,google/gemma-2-2b-it,gemma2-2b-it,overall,1050,270,0.257143,0.700165,14.504952,7.501582


In [68]:
SUMMARY_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_mcq_test_summary.csv"
)

final_summary.to_csv(
    SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Saved summary to:")
print(SUMMARY_PATH)

Saved summary to:
/content/drive/MyDrive/llm_benchmark/parsbench/gemma2-2b-it_parsinlu_mcq_test_summary.csv


Qwen 2.5 3B

In [69]:
unload_model()

Previous model unloaded.


In [70]:
MODEL_NAME = MODELS["qwen2.5-3b-instruct"]
MODEL_SLUG = "qwen2.5-3b-instruct"

load_model(MODEL_NAME)

CHECKPOINT_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_mcq_test_checkpoint.csv"
)

SUMMARY_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_mcq_test_summary.csv"
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-3B-Instruct
GPU: Tesla T4


In [71]:
smoke_eval_df = (
    eval_df
    .groupby(
        "category",
        group_keys=False
    )
    .head(10)
    .copy()
)

smoke_results_df = run_mcq_evaluation(
    smoke_eval_df,
    CHECKPOINT_PATH
)

No checkpoint found.
Total examples: 30
Already completed: 0
Remaining: 30
------------------------------------------------------------
30/30 completed | ID 710 | literature | target=4 | pred=3
Evaluation finished.


In [72]:
smoke_ids = set(
    smoke_eval_df["example_id"]
)

balanced_smoke = smoke_results_df[
    smoke_results_df["example_id"].isin(
        smoke_ids
    )
].copy()

summary = (
    balanced_smoke
    .groupby("category")
    .agg(
        questions=("example_id", "count"),
        correct=("correct", "sum"),
        accuracy=("correct", "mean"),
        avg_latency_sec=("latency_sec", "mean"),
        avg_tokens_per_sec=("tokens_per_sec", "mean"),
        peak_vram_gb=("peak_vram_gb", "max"),
    )
)

display(summary)

print(
    f"Overall balanced smoke accuracy: "
    f"{balanced_smoke['correct'].mean():.2%}"
)

,questions,correct,accuracy,avg_latency_sec,avg_tokens_per_sec,peak_vram_gb
category,,,,,,
common_knowledge,10,4,0.4,0.427752,7.963217,7.319394
literature,10,3,0.3,0.620332,6.081518,7.332457
math_and_logic,10,6,0.6,0.330281,8.575016,7.319774


Overall balanced smoke accuracy: 43.33%


In [73]:
results_df = run_mcq_evaluation(
    eval_df,
    CHECKPOINT_PATH
)

Loaded checkpoint: 30 rows
Total examples: 1050
Already completed: 30
Remaining: 1020
------------------------------------------------------------
1050/1050 completed | ID 1050 | literature | target=1 | pred=3
Evaluation finished.


In [74]:
successful = results_df[
    results_df["status"] == "success"
].copy()

print("Successful rows:", len(successful))
print("Errors:", (results_df["status"] == "error").sum())

Successful rows: 1050
Errors: 0


In [75]:
category_summary = (
    successful
    .groupby("category")
    .agg(
        questions=("example_id", "count"),
        correct=("correct", "sum"),
        accuracy=("correct", "mean"),
        avg_latency_sec=("latency_sec", "mean"),
        avg_tokens_per_sec=("tokens_per_sec", "mean"),
        peak_vram_gb=("peak_vram_gb", "max"),
    )
    .reset_index()
)

display(category_summary)

,category,questions,correct,accuracy,avg_latency_sec,avg_tokens_per_sec,peak_vram_gb
0,common_knowledge,350,118,0.337143,0.426553,9.672781,7.327033
1,literature,350,94,0.268571,0.335874,8.630971,7.339844
2,math_and_logic,350,98,0.280000,0.500544,9.263321,7.337164


In [76]:
overall_summary = pd.DataFrame([{
    "category": "overall",
    "questions": len(successful),
    "correct": int(successful["correct"].sum()),
    "accuracy": successful["correct"].mean(),
    "avg_latency_sec": successful["latency_sec"].mean(),
    "avg_tokens_per_sec": successful["tokens_per_sec"].mean(),
    "peak_vram_gb": successful["peak_vram_gb"].max(),
}])

In [77]:
final_summary = pd.concat(
    [
        category_summary,
        overall_summary
    ],
    ignore_index=True
)

final_summary.insert(
    0,
    "model_name",
    MODEL_NAME
)

final_summary.insert(
    1,
    "model_slug",
    MODEL_SLUG
)

display(final_summary)

,model_name,model_slug,category,questions,correct,accuracy,avg_latency_sec,avg_tokens_per_sec,peak_vram_gb
0,Qwen/Qwen2.5-3B-Instruct,qwen2.5-3b-instruct,common_knowledge,350,118,0.337143,0.426553,9.672781,7.327033
1,Qwen/Qwen2.5-3B-Instruct,qwen2.5-3b-instruct,literature,350,94,0.268571,0.335874,8.630971,7.339844
2,Qwen/Qwen2.5-3B-Instruct,qwen2.5-3b-instruct,math_and_logic,350,98,0.280000,0.500544,9.263321,7.337164
3,Qwen/Qwen2.5-3B-Instruct,qwen2.5-3b-instruct,overall,1050,310,0.295238,0.420990,9.189024,7.339844


In [78]:
SUMMARY_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_mcq_test_summary.csv"
)

final_summary.to_csv(
    SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Saved summary to:")
print(SUMMARY_PATH)

Saved summary to:
/content/drive/MyDrive/llm_benchmark/parsbench/qwen2.5-3b-instruct_parsinlu_mcq_test_summary.csv


Qwen 2.5 7B

In [81]:
unload_model()

Previous model unloaded.


In [82]:
MODEL_NAME = MODELS["qwen2.5-7b-instruct"]
MODEL_SLUG = "qwen2.5-7b-instruct"

load_model(MODEL_NAME)

CHECKPOINT_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_mcq_test_checkpoint.csv"
)

SUMMARY_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_mcq_test_summary.csv"
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-7B-Instruct
GPU: Tesla T4


In [83]:
smoke_eval_df = (
    eval_df
    .groupby(
        "category",
        group_keys=False
    )
    .head(10)
    .copy()
)

smoke_results_df = run_mcq_evaluation(
    smoke_eval_df,
    CHECKPOINT_PATH
)

No checkpoint found.
Total examples: 30
Already completed: 0
Remaining: 30
------------------------------------------------------------
30/30 completed | ID 710 | literature | target=4 | pred=4
Evaluation finished.


In [84]:
smoke_ids = set(
    smoke_eval_df["example_id"]
)

balanced_smoke = smoke_results_df[
    smoke_results_df["example_id"].isin(
        smoke_ids
    )
].copy()

summary = (
    balanced_smoke
    .groupby("category")
    .agg(
        questions=("example_id", "count"),
        correct=("correct", "sum"),
        accuracy=("correct", "mean"),
        avg_latency_sec=("latency_sec", "mean"),
        avg_tokens_per_sec=("tokens_per_sec", "mean"),
        peak_vram_gb=("peak_vram_gb", "max"),
    )
)

display(summary)

print(
    f"Overall balanced smoke accuracy: "
    f"{balanced_smoke['correct'].mean():.2%}"
)

,questions,correct,accuracy,avg_latency_sec,avg_tokens_per_sec,peak_vram_gb
category,,,,,,
common_knowledge,10,6,0.6,1.090366,1.902318,10.683231
literature,10,4,0.4,0.625663,3.344187,10.705588
math_and_logic,10,6,0.6,0.518905,3.896256,10.683921


Overall balanced smoke accuracy: 53.33%


In [85]:
results_df = run_mcq_evaluation(
    eval_df,
    CHECKPOINT_PATH
)

Loaded checkpoint: 30 rows
Total examples: 1050
Already completed: 30
Remaining: 1020
------------------------------------------------------------
1050/1050 completed | ID 1050 | literature | target=1 | pred=1
Evaluation finished.


In [86]:
successful = results_df[
    results_df["status"] == "success"
].copy()

print("Successful rows:", len(successful))
print("Errors:", (results_df["status"] == "error").sum())

Successful rows: 1050
Errors: 0


In [87]:
category_summary = (
    successful
    .groupby("category")
    .agg(
        questions=("example_id", "count"),
        correct=("correct", "sum"),
        accuracy=("correct", "mean"),
        avg_latency_sec=("latency_sec", "mean"),
        avg_tokens_per_sec=("tokens_per_sec", "mean"),
        peak_vram_gb=("peak_vram_gb", "max"),
    )
    .reset_index()
)

display(category_summary)

,category,questions,correct,accuracy,avg_latency_sec,avg_tokens_per_sec,peak_vram_gb
0,common_knowledge,350,158,0.451429,0.488600,4.237913,10.695005
1,literature,350,117,0.334286,0.487027,4.207354,10.718669
2,math_and_logic,350,149,0.425714,0.510561,3.997492,10.714429


In [88]:
overall_summary = pd.DataFrame([{
    "category": "overall",
    "questions": len(successful),
    "correct": int(successful["correct"].sum()),
    "accuracy": successful["correct"].mean(),
    "avg_latency_sec": successful["latency_sec"].mean(),
    "avg_tokens_per_sec": successful["tokens_per_sec"].mean(),
    "peak_vram_gb": successful["peak_vram_gb"].max(),
}])

In [89]:
final_summary = pd.concat(
    [
        category_summary,
        overall_summary
    ],
    ignore_index=True
)

final_summary.insert(
    0,
    "model_name",
    MODEL_NAME
)

final_summary.insert(
    1,
    "model_slug",
    MODEL_SLUG
)

display(final_summary)

,model_name,model_slug,category,questions,correct,accuracy,avg_latency_sec,avg_tokens_per_sec,peak_vram_gb
0,Qwen/Qwen2.5-7B-Instruct,qwen2.5-7b-instruct,common_knowledge,350,158,0.451429,0.488600,4.237913,10.695005
1,Qwen/Qwen2.5-7B-Instruct,qwen2.5-7b-instruct,literature,350,117,0.334286,0.487027,4.207354,10.718669
2,Qwen/Qwen2.5-7B-Instruct,qwen2.5-7b-instruct,math_and_logic,350,149,0.425714,0.510561,3.997492,10.714429
3,Qwen/Qwen2.5-7B-Instruct,qwen2.5-7b-instruct,overall,1050,424,0.403810,0.495396,4.147586,10.718669


In [90]:
SUMMARY_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_mcq_test_summary.csv"
)

final_summary.to_csv(
    SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Saved summary to:")
print(SUMMARY_PATH)

Saved summary to:
/content/drive/MyDrive/llm_benchmark/parsbench/qwen2.5-7b-instruct_parsinlu_mcq_test_summary.csv


ParsiNLU Reading Comprehension

In [91]:
from datasets import load_dataset

rc_ds = load_dataset(
    "ParsBench/parsinlu-reading-comprehension-alpaca-style",
    split="train"
)

rc_df = rc_ds.to_pandas().reset_index(drop=True)

rc_df["example_id"] = range(
    1,
    len(rc_df) + 1
)

print("Total RC examples:", len(rc_df))
print(rc_df.columns.tolist())

display(rc_df.head(3))

README.md:   0%|          | 0.00/331 [00:00<?, ?B/s]

dataset.json:   0%|          | 0.00/857k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/600 [00:00<?, ? examples/s]

Total RC examples: 600
['instruction', 'output', 'input', 'example_id']


,instruction,output,input,example_id
0,در ادامه به شما یک متن فارسی و یک سوال نشان دا...,جواب: نزدیکی روابط لهستان به آمریکا,متن: '''لهستان یکی از وفادارترین متحدان آمریکا...,1
1,در ادامه به شما یک متن فارسی و یک سوال نشان دا...,جواب: ایلیا سلیمان,متن: '''بهشت حتماً همین است (انگلیسی: It Must ...,2
2,در ادامه به شما یک متن فارسی و یک سوال نشان دا...,جواب: ۱۷ تن از امیران ارتش,متن: '''از سال ۱۳۳۷ تا سال ۱۳۵۵، ۱۷ تن از امیر...,3


In [92]:
def build_rc_prompt(row):
    instruction = str(row["instruction"]).strip()
    input_text = str(row["input"]).strip()

    return f"""{instruction}

{input_text}"""

In [93]:
print(build_rc_prompt(rc_df.iloc[0]))
print("\nREFERENCE:")
print(rc_df.iloc[0]["output"])

در ادامه به شما یک متن فارسی و یک سوال نشان داده می شود. شما باید برای سوال یک پاسخ بنویسید. سعی کنید پاسخ های خود را تا حد ممکن کوتاه بدهید.

متن: '''لهستان یکی از وفادارترین متحدان آمریکا در اروپای مرکزی و شرقی است. این کشور هم‌گام با اتحادیه اروپا از توافق هسته‌ای ایران با قدرتهای جهانی موسوم به برجام دفاع کرده اما همیشه با یک تبصره؛ اینکه دغدغه‌های آمریکا در برابر ایران را درک می‌کند.  همچنین به دلیل روابط گرم بین ورشو و واشنگتن، دولت لهستان بارها به اتحادیه اروپا پیشنهاد داده که حاضر است به عنوان میانجی بر سر مسئله برجام بین آمریکا و اتحادیه اروپا نقش ایفا کند.  اما نزدیکی روابط لهستان به آمریکا که می‌تواند منجر به میزبانی این کنفرانس شده باشد دلایل متعددی دارد.'''
سوال: '''چرا آمریکا لهستان را انتخاب کرد؟'''

REFERENCE:
جواب: نزدیکی روابط لهستان به آمریکا


In [94]:
@torch.inference_mode()
def generate_rc_answer(prompt):
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    chat_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        chat_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    torch.cuda.synchronize()
    start = time.perf_counter()

    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    torch.cuda.synchronize()
    latency = time.perf_counter() - start

    generated_ids = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    completion = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    output_tokens = len(generated_ids)

    return completion, latency, output_tokens

In [95]:
!pip install -q sentence-transformers

In [96]:
from sentence_transformers import SentenceTransformer, util

semantic_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [97]:
import re
import unicodedata
from collections import Counter


def normalize_fa_text(text):
    if text is None:
        return ""

    text = str(text)
    text = unicodedata.normalize("NFKC", text)

    text = (
        text
        .replace("ي", "ی")
        .replace("ك", "ک")
        .replace("\u200c", " ")
    )

    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip().lower()


def token_f1(prediction, reference):
    pred_tokens = normalize_fa_text(prediction).split()
    ref_tokens = normalize_fa_text(reference).split()

    if not pred_tokens and not ref_tokens:
        return 1.0

    if not pred_tokens or not ref_tokens:
        return 0.0

    common = Counter(pred_tokens) & Counter(ref_tokens)
    common_count = sum(common.values())

    if common_count == 0:
        return 0.0

    precision = common_count / len(pred_tokens)
    recall = common_count / len(ref_tokens)

    return (
        2 * precision * recall
        / (precision + recall)
    )

In [98]:
def clean_answer(text):
    text = str(text).strip()

    text = re.sub(
        r"^\s*(?:جواب|پاسخ)\s*[:：]\s*",
        "",
        text
    )

    return text.strip()

In [99]:
def semantic_similarity(prediction, reference):
    prediction = clean_answer(prediction)
    reference = clean_answer(reference)

    embeddings = semantic_model.encode(
        [prediction, reference],
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    score = util.cos_sim(
        embeddings[0],
        embeddings[1]
    ).item()

    return float(score)

In [100]:
def rc_scores(prediction, reference):
    pred_clean = clean_answer(prediction)
    ref_clean = clean_answer(reference)

    f1 = token_f1(
        pred_clean,
        ref_clean
    )

    semantic = semantic_similarity(
        pred_clean,
        ref_clean
    )

    return {
        "f1": f1,
        "semantic_similarity": semantic
    }

In [101]:
reference = "جواب: نزدیکی روابط لهستان به آمریکا"
prediction = "نزدیکی روابط لهستان با آمریکا"

print(rc_scores(prediction, reference))

{'f1': 0.8000000000000002, 'semantic_similarity': 0.9958493709564209}


### Llama/ Dorna

In [150]:
unload_model()

Previous model unloaded.


In [151]:
MODEL_NAME = MODELS["dorna2-8b"]
MODEL_SLUG = "dorna2-8b"

load_model(MODEL_NAME)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded: PartAI/Dorna2-Llama3.1-8B-Instruct
GPU: Tesla T4


In [152]:
CHECKPOINT_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_mcq_test_checkpoint.csv"
)

SUMMARY_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_mcq_test_summary.csv"
)

In [153]:
RC_CHECKPOINT_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_rc_checkpoint.csv"
)

RC_SCORED_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_rc_scored.csv"
)

RC_SUMMARY_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_rc_summary.csv"
)

print("Checkpoint:", RC_CHECKPOINT_PATH)
print("Scored results:", RC_SCORED_PATH)
print("Summary:", RC_SUMMARY_PATH)

Checkpoint: /content/drive/MyDrive/llm_benchmark/parsbench/dorna2-8b_parsinlu_rc_checkpoint.csv
Scored results: /content/drive/MyDrive/llm_benchmark/parsbench/dorna2-8b_parsinlu_rc_scored.csv
Summary: /content/drive/MyDrive/llm_benchmark/parsbench/dorna2-8b_parsinlu_rc_summary.csv


In [154]:
import os
import pandas as pd


def load_rc_checkpoint(path):
    if not path.exists():
        print("No RC checkpoint found.")
        return {}

    df = pd.read_csv(path)

    results = {
        int(row["example_id"]): row.to_dict()
        for _, row in df.iterrows()
    }

    print(f"Loaded RC checkpoint: {len(results)} rows")

    return results


def save_rc_checkpoint(results, path):
    if not results:
        return

    df = (
        pd.DataFrame(list(results.values()))
        .sort_values("example_id")
    )

    tmp_path = str(path) + ".tmp"

    df.to_csv(
        tmp_path,
        index=False,
        encoding="utf-8-sig"
    )

    os.replace(tmp_path, path)

In [155]:
def run_rc_evaluation(
    dataset,
    checkpoint_path,
    limit=None
):
    results = load_rc_checkpoint(
        checkpoint_path
    )

    successful_ids = {
        example_id
        for example_id, result in results.items()
        if result.get("status") == "success"
    }

    if limit is not None:
        work_df = dataset.iloc[:limit].copy()
    else:
        work_df = dataset.copy()

    work_ids = set(
        work_df["example_id"].astype(int)
    )

    already_done = len(
        successful_ids.intersection(work_ids)
    )

    total = len(work_df)

    print(f"Total examples: {total}")
    print(f"Already completed: {already_done}")
    print(f"Remaining: {total - already_done}")
    print("-" * 60)

    for _, row in work_df.iterrows():

        example_id = int(
            row["example_id"]
        )

        if example_id in successful_ids:
            continue

        prompt = build_rc_prompt(row)

        reference = str(
            row["output"]
        ).strip()

        try:
            torch.cuda.reset_peak_memory_stats()

            (
                completion,
                latency,
                output_tokens
            ) = generate_rc_answer(prompt)

            clean_prediction = clean_answer(
                completion
            )

            clean_reference = clean_answer(
                reference
            )

            f1 = token_f1(
                clean_prediction,
                clean_reference
            )

            peak_vram_gb = (
                torch.cuda.max_memory_allocated()
                / 1024**3
            )

            tokens_per_sec = (
                output_tokens / latency
                if latency > 0
                else None
            )

            results[example_id] = {
                "model_name": MODEL_NAME,
                "model_slug": MODEL_SLUG,

                "example_id": example_id,

                "instruction": row["instruction"],
                "input": row["input"],

                "reference": reference,
                "prediction": completion,

                "clean_reference": clean_reference,
                "clean_prediction": clean_prediction,

                "f1": f1,

                "latency_sec": latency,
                "output_tokens": output_tokens,
                "tokens_per_sec": tokens_per_sec,
                "peak_vram_gb": peak_vram_gb,

                "status": "success",
                "error": "",
            }

            successful_ids.add(
                example_id
            )

        except Exception as e:

            results[example_id] = {
                "model_name": MODEL_NAME,
                "model_slug": MODEL_SLUG,

                "example_id": example_id,

                "instruction": row["instruction"],
                "input": row["input"],

                "reference": reference,
                "prediction": "",

                "clean_reference":
                    clean_answer(reference),

                "clean_prediction": "",

                "f1": 0.0,

                "latency_sec": None,
                "output_tokens": None,
                "tokens_per_sec": None,
                "peak_vram_gb": None,

                "status": "error",
                "error": str(e),
            }

            print(
                f"\nERROR example {example_id}: {e}"
            )

        # Save after EVERY example
        save_rc_checkpoint(
            results,
            checkpoint_path
        )

        completed = len(
            successful_ids.intersection(
                work_ids
            )
        )

        print(
            f"\r{completed}/{total} completed | "
            f"ID {example_id} | "
            f"F1={results[example_id]['f1']:.3f}",
            end=""
        )

    print("\nRC evaluation finished.")

    return (
        pd.DataFrame(
            list(results.values())
        )
        .sort_values("example_id")
    )

In [156]:
rc_smoke_df = run_rc_evaluation(
    rc_df,
    RC_CHECKPOINT_PATH,
    limit=10
)

No RC checkpoint found.
Total examples: 10
Already completed: 0
Remaining: 10
------------------------------------------------------------
10/10 completed | ID 10 | F1=0.130
RC evaluation finished.


In [157]:
display(
    rc_smoke_df[
        [
            "example_id",
            "reference",
            "prediction",
            "f1",
            "latency_sec"
        ]
    ].head(10)
)

print(
    "Mean smoke F1:",
    rc_smoke_df[
        rc_smoke_df["example_id"] <= 10
    ]["f1"].mean()
)

,example_id,reference,prediction,f1,latency_sec
0,1,جواب: نزدیکی روابط لهستان به آمریکا,لهستان به دلیل وفاداری و نزدیکی خود به آمریکا،...,0.204082,4.332458
1,2,جواب: ایلیا سلیمان,ایلیا سلیمان,1.000000,0.767936
2,3,جواب: ۱۷ تن از امیران ارتش,در سال‌های ۱۳۳۷ تا ۱۳۵۵، ۱۷ تن از امیران ارتش ...,0.227273,6.330350
3,4,جواب: مشخصه بیماران مبتلا به اختلال شخصیت بدگم...,پارانوئید یک اختلال شخصیت است که با شکاکیت، بی...,0.366667,6.991046
4,5,جواب: اعتقاد بر این است که سروتونین که یک انتق...,خودکشی یک پدیده پیچیده است که تحت تاثیر عوامل ...,0.237288,5.367850
5,6,جواب: عفونت باکتریایی، انگلی یا ویروسی,عفونت باکتریایی، انگلی یا ویروسی,1.000000,1.241376
6,7,جواب: طیف وسیعی از میکروارگانیسم‌های بیماری‌زا...,عوامل مختلفی می‌توانند آب را آلوده کنند، از جم...,0.620690,4.502047
7,8,جواب: اگر شما در حال تلاش برای چاق شدن هستید، ...,مهم‌ترین ماده مغذی برای چاقی سالم، پروتئین است...,0.161290,5.151094
8,9,جواب: گوش‌,گوش انسان تا آخر عمر به رشد ادامه می‌دهد.,0.181818,1.171402
9,10,جواب: شماره‌های ایرانسل منطقه بندی ندارند,برای فهمیدن اینکه شماره ایرانسل یک شماره دائمی...,0.130435,4.501858


Mean smoke F1: 0.4129542104365612


In [158]:
results_df = run_rc_evaluation(
    rc_df,
    RC_CHECKPOINT_PATH
)

Loaded RC checkpoint: 10 rows
Total examples: 600
Already completed: 10
Remaining: 590
------------------------------------------------------------
600/600 completed | ID 600 | F1=0.813
RC evaluation finished.


In [159]:
 results_df = pd.read_csv(
    RC_CHECKPOINT_PATH
)

successful_mask = (
    results_df["status"] == "success"
)

successful = results_df[
    successful_mask
].copy()

print(
    "Responses to score:",
    len(successful)
)

Responses to score: 600


In [160]:
predictions = (
    successful["clean_prediction"]
    .fillna("")
    .astype(str)
    .tolist()
)

references = (
    successful["clean_reference"]
    .fillna("")
    .astype(str)
    .tolist()
)

pred_embeddings = semantic_model.encode(
    predictions,
    batch_size=32,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

ref_embeddings = semantic_model.encode(
    references,
    batch_size=32,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

In [161]:
similarities = (
    torch.sum(
        pred_embeddings * ref_embeddings,
        dim=1
    )
    .cpu()
    .numpy()
)

successful[
    "semantic_similarity"
] = similarities

In [162]:
similarity_map = dict(
    zip(
        successful["example_id"],
        successful["semantic_similarity"]
    )
)

results_df[
    "semantic_similarity"
] = results_df[
    "example_id"
].map(
    similarity_map
)

In [163]:
results_df.to_csv(
    RC_SCORED_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Saved scored RC results:")
print(RC_SCORED_PATH)

Saved scored RC results:
/content/drive/MyDrive/llm_benchmark/parsbench/dorna2-8b_parsinlu_rc_scored.csv


In [164]:
results_df.to_csv(
    RC_SCORED_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Saved scored RC results:")
print(RC_SCORED_PATH)

Saved scored RC results:
/content/drive/MyDrive/llm_benchmark/parsbench/dorna2-8b_parsinlu_rc_scored.csv


In [165]:
successful = results_df[
    results_df["status"] == "success"
].copy()

rc_summary = pd.DataFrame([{
    "model_name": MODEL_NAME,
    "model_slug": MODEL_SLUG,

    "questions": len(successful),

    "mean_f1":
        successful["f1"].mean(),

    "median_f1":
        successful["f1"].median(),

    "mean_semantic_similarity":
        successful[
            "semantic_similarity"
        ].mean(),

    "median_semantic_similarity":
        successful[
            "semantic_similarity"
        ].median(),

    "avg_latency_sec":
        successful[
            "latency_sec"
        ].mean(),

    "avg_output_tokens":
        successful[
            "output_tokens"
        ].mean(),

    "avg_tokens_per_sec":
        successful[
            "tokens_per_sec"
        ].mean(),

    "peak_vram_gb":
        successful[
            "peak_vram_gb"
        ].max(),

    "errors":
        int(
            (results_df["status"] == "error")
            .sum()
        ),
}])

display(rc_summary)

,model_name,model_slug,questions,mean_f1,median_f1,mean_semantic_similarity,median_semantic_similarity,avg_latency_sec,avg_output_tokens,avg_tokens_per_sec,peak_vram_gb,errors
0,PartAI/Dorna2-Llama3.1-8B-Instruct,dorna2-8b,600,0.392814,0.333333,0.663621,0.675036,2.69127,35.216667,11.953065,12.245471,0


In [166]:
results_df = pd.read_csv(
    RC_SCORED_PATH
)

def exact_match(prediction, reference):
    pred = normalize_fa_text(
        clean_answer(prediction)
    )

    ref = normalize_fa_text(
        clean_answer(reference)
    )

    return int(pred == ref)


results_df["exact_match"] = results_df.apply(
    lambda row: exact_match(
        row["prediction"],
        row["reference"]
    )
    if row["status"] == "success"
    else 0,
    axis=1
)

results_df.to_csv(
    RC_SCORED_PATH,
    index=False,
    encoding="utf-8-sig"
)

successful = results_df[
    results_df["status"] == "success"
].copy()

print(
    f"Exact Match: "
    f"{successful['exact_match'].mean():.4f}"
)

print(
    f"Mean F1: "
    f"{successful['f1'].mean():.4f}"
)

print(
    f"Mean Semantic Similarity: "
    f"{successful['semantic_similarity'].mean():.4f}"
)

Exact Match: 0.0967
Mean F1: 0.3928
Mean Semantic Similarity: 0.6636


In [167]:
rc_summary = pd.DataFrame([{
    "model_name": MODEL_NAME,
    "model_slug": MODEL_SLUG,

    "questions": len(successful),

    "exact_match":
        successful["exact_match"].mean(),

    "mean_f1":
        successful["f1"].mean(),

    "median_f1":
        successful["f1"].median(),

    "mean_semantic_similarity":
        successful[
            "semantic_similarity"
        ].mean(),

    "median_semantic_similarity":
        successful[
            "semantic_similarity"
        ].median(),

    "avg_latency_sec":
        successful[
            "latency_sec"
        ].mean(),

    "avg_output_tokens":
        successful[
            "output_tokens"
        ].mean(),

    "avg_tokens_per_sec":
        successful[
            "tokens_per_sec"
        ].mean(),

    "peak_vram_gb":
        successful[
            "peak_vram_gb"
        ].max(),

    "errors":
        int(
            (results_df["status"] == "error")
            .sum()
        ),
}])

rc_summary.to_csv(
    RC_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

display(rc_summary)

,model_name,model_slug,questions,exact_match,mean_f1,median_f1,mean_semantic_similarity,median_semantic_similarity,avg_latency_sec,avg_output_tokens,avg_tokens_per_sec,peak_vram_gb,errors
0,PartAI/Dorna2-Llama3.1-8B-Instruct,dorna2-8b,600,0.096667,0.392814,0.333333,0.663621,0.675036,2.69127,35.216667,11.953065,12.245471,0


### Gemma 2

In [133]:
unload_model()

Previous model unloaded.


In [134]:
MODEL_NAME = MODELS["gemma2-2b-it"]
MODEL_SLUG = "gemma2-2b-it"

load_model(MODEL_NAME)

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Loaded: google/gemma-2-2b-it
GPU: Tesla T4


In [135]:
CHECKPOINT_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_mcq_test_checkpoint.csv"
)

SUMMARY_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_mcq_test_summary.csv"
)

In [136]:
RC_CHECKPOINT_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_rc_checkpoint.csv"
)

RC_SCORED_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_rc_scored.csv"
)

RC_SUMMARY_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_rc_summary.csv"
)

print("Checkpoint:", RC_CHECKPOINT_PATH)
print("Scored results:", RC_SCORED_PATH)
print("Summary:", RC_SUMMARY_PATH)

Checkpoint: /content/drive/MyDrive/llm_benchmark/parsbench/gemma2-2b-it_parsinlu_rc_checkpoint.csv
Scored results: /content/drive/MyDrive/llm_benchmark/parsbench/gemma2-2b-it_parsinlu_rc_scored.csv
Summary: /content/drive/MyDrive/llm_benchmark/parsbench/gemma2-2b-it_parsinlu_rc_summary.csv


In [137]:
import os
import pandas as pd


def load_rc_checkpoint(path):
    if not path.exists():
        print("No RC checkpoint found.")
        return {}

    df = pd.read_csv(path)

    results = {
        int(row["example_id"]): row.to_dict()
        for _, row in df.iterrows()
    }

    print(f"Loaded RC checkpoint: {len(results)} rows")

    return results


def save_rc_checkpoint(results, path):
    if not results:
        return

    df = (
        pd.DataFrame(list(results.values()))
        .sort_values("example_id")
    )

    tmp_path = str(path) + ".tmp"

    df.to_csv(
        tmp_path,
        index=False,
        encoding="utf-8-sig"
    )

    os.replace(tmp_path, path)

In [138]:
def run_rc_evaluation(
    dataset,
    checkpoint_path,
    limit=None
):
    results = load_rc_checkpoint(
        checkpoint_path
    )

    successful_ids = {
        example_id
        for example_id, result in results.items()
        if result.get("status") == "success"
    }

    if limit is not None:
        work_df = dataset.iloc[:limit].copy()
    else:
        work_df = dataset.copy()

    work_ids = set(
        work_df["example_id"].astype(int)
    )

    already_done = len(
        successful_ids.intersection(work_ids)
    )

    total = len(work_df)

    print(f"Total examples: {total}")
    print(f"Already completed: {already_done}")
    print(f"Remaining: {total - already_done}")
    print("-" * 60)

    for _, row in work_df.iterrows():

        example_id = int(
            row["example_id"]
        )

        if example_id in successful_ids:
            continue

        prompt = build_rc_prompt(row)

        reference = str(
            row["output"]
        ).strip()

        try:
            torch.cuda.reset_peak_memory_stats()

            (
                completion,
                latency,
                output_tokens
            ) = generate_rc_answer(prompt)

            clean_prediction = clean_answer(
                completion
            )

            clean_reference = clean_answer(
                reference
            )

            f1 = token_f1(
                clean_prediction,
                clean_reference
            )

            peak_vram_gb = (
                torch.cuda.max_memory_allocated()
                / 1024**3
            )

            tokens_per_sec = (
                output_tokens / latency
                if latency > 0
                else None
            )

            results[example_id] = {
                "model_name": MODEL_NAME,
                "model_slug": MODEL_SLUG,

                "example_id": example_id,

                "instruction": row["instruction"],
                "input": row["input"],

                "reference": reference,
                "prediction": completion,

                "clean_reference": clean_reference,
                "clean_prediction": clean_prediction,

                "f1": f1,

                "latency_sec": latency,
                "output_tokens": output_tokens,
                "tokens_per_sec": tokens_per_sec,
                "peak_vram_gb": peak_vram_gb,

                "status": "success",
                "error": "",
            }

            successful_ids.add(
                example_id
            )

        except Exception as e:

            results[example_id] = {
                "model_name": MODEL_NAME,
                "model_slug": MODEL_SLUG,

                "example_id": example_id,

                "instruction": row["instruction"],
                "input": row["input"],

                "reference": reference,
                "prediction": "",

                "clean_reference":
                    clean_answer(reference),

                "clean_prediction": "",

                "f1": 0.0,

                "latency_sec": None,
                "output_tokens": None,
                "tokens_per_sec": None,
                "peak_vram_gb": None,

                "status": "error",
                "error": str(e),
            }

            print(
                f"\nERROR example {example_id}: {e}"
            )

        # Save after EVERY example
        save_rc_checkpoint(
            results,
            checkpoint_path
        )

        completed = len(
            successful_ids.intersection(
                work_ids
            )
        )

        print(
            f"\r{completed}/{total} completed | "
            f"ID {example_id} | "
            f"F1={results[example_id]['f1']:.3f}",
            end=""
        )

    print("\nRC evaluation finished.")

    return (
        pd.DataFrame(
            list(results.values())
        )
        .sort_values("example_id")
    )

In [139]:
rc_smoke_df = run_rc_evaluation(
    rc_df,
    RC_CHECKPOINT_PATH,
    limit=10
)

No RC checkpoint found.
Total examples: 10
Already completed: 0
Remaining: 10
------------------------------------------------------------
10/10 completed | ID 10 | F1=0.143
RC evaluation finished.


In [140]:
display(
    rc_smoke_df[
        [
            "example_id",
            "reference",
            "prediction",
            "f1",
            "latency_sec"
        ]
    ].head(10)
)

print(
    "Mean smoke F1:",
    rc_smoke_df[
        rc_smoke_df["example_id"] <= 10
    ]["f1"].mean()
)

,example_id,reference,prediction,f1,latency_sec
0,1,جواب: نزدیکی روابط لهستان به آمریکا,متن متن بیان می کند که آمریکا لهستان را به عنو...,0.122449,4.310152
1,2,جواب: ایلیا سلیمان,ایلیا سلیمان,1.000000,0.729763
2,3,جواب: ۱۷ تن از امیران ارتش,امیران ارتش در ایران در سال‌های ۱۳۳۷ تا ۱۳۵۵، ...,0.250000,2.516597
3,4,جواب: مشخصه بیماران مبتلا به اختلال شخصیت بدگم...,پارانوئید یک اختلال شخصیت است.,0.285714,0.864634
4,5,جواب: اعتقاد بر این است که سروتونین که یک انتق...,متن متن به طور کلی به عوامل مختلفی که در خودکش...,0.244898,2.210751
5,6,جواب: عفونت باکتریایی، انگلی یا ویروسی,عفونت باکتریایی، انگلی یا ویروسی می تواند به د...,0.384615,2.378160
6,7,جواب: طیف وسیعی از میکروارگانیسم‌های بیماری‌زا...,آلاینده‌های مختلفی مانند میکروارگانیسم‌های بیم...,0.701754,4.294821
7,8,جواب: اگر شما در حال تلاش برای چاق شدن هستید، ...,پروتئین,0.080000,0.513021
8,9,جواب: گوش‌,گوش,1.000000,0.432790
9,10,جواب: شماره‌های ایرانسل منطقه بندی ندارند,برای اینکه بفهمیم شماره ایرانسل متعلق به کدوم ...,0.142857,2.377575


Mean smoke F1: 0.4212288137927236


In [141]:
results_df = run_rc_evaluation(
    rc_df,
    RC_CHECKPOINT_PATH
)

Loaded RC checkpoint: 10 rows
Total examples: 600
Already completed: 10
Remaining: 590
------------------------------------------------------------
600/600 completed | ID 600 | F1=0.867
RC evaluation finished.


In [142]:
 results_df = pd.read_csv(
    RC_CHECKPOINT_PATH
)

successful_mask = (
    results_df["status"] == "success"
)

successful = results_df[
    successful_mask
].copy()

print(
    "Responses to score:",
    len(successful)
)

Responses to score: 600


In [143]:
predictions = (
    successful["clean_prediction"]
    .fillna("")
    .astype(str)
    .tolist()
)

references = (
    successful["clean_reference"]
    .fillna("")
    .astype(str)
    .tolist()
)

pred_embeddings = semantic_model.encode(
    predictions,
    batch_size=32,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

ref_embeddings = semantic_model.encode(
    references,
    batch_size=32,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

In [144]:
similarities = (
    torch.sum(
        pred_embeddings * ref_embeddings,
        dim=1
    )
    .cpu()
    .numpy()
)

successful[
    "semantic_similarity"
] = similarities

In [145]:
similarity_map = dict(
    zip(
        successful["example_id"],
        successful["semantic_similarity"]
    )
)

results_df[
    "semantic_similarity"
] = results_df[
    "example_id"
].map(
    similarity_map
)

In [146]:
results_df.to_csv(
    RC_SCORED_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Saved scored RC results:")
print(RC_SCORED_PATH)

Saved scored RC results:
/content/drive/MyDrive/llm_benchmark/parsbench/gemma2-2b-it_parsinlu_rc_scored.csv


In [147]:
successful = results_df[
    results_df["status"] == "success"
].copy()

rc_summary = pd.DataFrame([{
    "model_name": MODEL_NAME,
    "model_slug": MODEL_SLUG,

    "questions": len(successful),

    "mean_f1":
        successful["f1"].mean(),

    "median_f1":
        successful["f1"].median(),

    "mean_semantic_similarity":
        successful[
            "semantic_similarity"
        ].mean(),

    "median_semantic_similarity":
        successful[
            "semantic_similarity"
        ].median(),

    "avg_latency_sec":
        successful[
            "latency_sec"
        ].mean(),

    "avg_output_tokens":
        successful[
            "output_tokens"
        ].mean(),

    "avg_tokens_per_sec":
        successful[
            "tokens_per_sec"
        ].mean(),

    "peak_vram_gb":
        successful[
            "peak_vram_gb"
        ].max(),

    "errors":
        int(
            (results_df["status"] == "error")
            .sum()
        ),
}])

display(rc_summary)

,model_name,model_slug,questions,mean_f1,median_f1,mean_semantic_similarity,median_semantic_similarity,avg_latency_sec,avg_output_tokens,avg_tokens_per_sec,peak_vram_gb,errors
0,google/gemma-2-2b-it,gemma2-2b-it,600,0.480498,0.462477,0.712326,0.738642,1.570797,23.073333,14.418636,8.695781,0


In [148]:
results_df = pd.read_csv(
    RC_SCORED_PATH
)

def exact_match(prediction, reference):
    pred = normalize_fa_text(
        clean_answer(prediction)
    )

    ref = normalize_fa_text(
        clean_answer(reference)
    )

    return int(pred == ref)


results_df["exact_match"] = results_df.apply(
    lambda row: exact_match(
        row["prediction"],
        row["reference"]
    )
    if row["status"] == "success"
    else 0,
    axis=1
)

results_df.to_csv(
    RC_SCORED_PATH,
    index=False,
    encoding="utf-8-sig"
)

successful = results_df[
    results_df["status"] == "success"
].copy()

print(
    f"Exact Match: "
    f"{successful['exact_match'].mean():.4f}"
)

print(
    f"Mean F1: "
    f"{successful['f1'].mean():.4f}"
)

print(
    f"Mean Semantic Similarity: "
    f"{successful['semantic_similarity'].mean():.4f}"
)

Exact Match: 0.1517
Mean F1: 0.4805
Mean Semantic Similarity: 0.7123


In [149]:
rc_summary = pd.DataFrame([{
    "model_name": MODEL_NAME,
    "model_slug": MODEL_SLUG,

    "questions": len(successful),

    "exact_match":
        successful["exact_match"].mean(),

    "mean_f1":
        successful["f1"].mean(),

    "median_f1":
        successful["f1"].median(),

    "mean_semantic_similarity":
        successful[
            "semantic_similarity"
        ].mean(),

    "median_semantic_similarity":
        successful[
            "semantic_similarity"
        ].median(),

    "avg_latency_sec":
        successful[
            "latency_sec"
        ].mean(),

    "avg_output_tokens":
        successful[
            "output_tokens"
        ].mean(),

    "avg_tokens_per_sec":
        successful[
            "tokens_per_sec"
        ].mean(),

    "peak_vram_gb":
        successful[
            "peak_vram_gb"
        ].max(),

    "errors":
        int(
            (results_df["status"] == "error")
            .sum()
        ),
}])

rc_summary.to_csv(
    RC_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

display(rc_summary)

,model_name,model_slug,questions,exact_match,mean_f1,median_f1,mean_semantic_similarity,median_semantic_similarity,avg_latency_sec,avg_output_tokens,avg_tokens_per_sec,peak_vram_gb,errors
0,google/gemma-2-2b-it,gemma2-2b-it,600,0.151667,0.480498,0.462477,0.712326,0.738642,1.570797,23.073333,14.418636,8.695781,0


### Qwen 2.5 3B

In [117]:
unload_model()

Previous model unloaded.


In [118]:
MODEL_NAME = MODELS["qwen2.5-3b-instruct"]
MODEL_SLUG = "qwen2.5-3b-instruct"

load_model(MODEL_NAME)

CHECKPOINT_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_mcq_test_checkpoint.csv"
)

SUMMARY_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_mcq_test_summary.csv"
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loaded: Qwen/Qwen2.5-3B-Instruct
GPU: Tesla T4


In [119]:
RC_CHECKPOINT_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_rc_checkpoint.csv"
)

RC_SCORED_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_rc_scored.csv"
)

RC_SUMMARY_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_rc_summary.csv"
)

print("Checkpoint:", RC_CHECKPOINT_PATH)
print("Scored results:", RC_SCORED_PATH)
print("Summary:", RC_SUMMARY_PATH)

Checkpoint: /content/drive/MyDrive/llm_benchmark/parsbench/qwen2.5-3b-instruct_parsinlu_rc_checkpoint.csv
Scored results: /content/drive/MyDrive/llm_benchmark/parsbench/qwen2.5-3b-instruct_parsinlu_rc_scored.csv
Summary: /content/drive/MyDrive/llm_benchmark/parsbench/qwen2.5-3b-instruct_parsinlu_rc_summary.csv


In [120]:
import os
import pandas as pd


def load_rc_checkpoint(path):
    if not path.exists():
        print("No RC checkpoint found.")
        return {}

    df = pd.read_csv(path)

    results = {
        int(row["example_id"]): row.to_dict()
        for _, row in df.iterrows()
    }

    print(f"Loaded RC checkpoint: {len(results)} rows")

    return results


def save_rc_checkpoint(results, path):
    if not results:
        return

    df = (
        pd.DataFrame(list(results.values()))
        .sort_values("example_id")
    )

    tmp_path = str(path) + ".tmp"

    df.to_csv(
        tmp_path,
        index=False,
        encoding="utf-8-sig"
    )

    os.replace(tmp_path, path)

In [121]:
def run_rc_evaluation(
    dataset,
    checkpoint_path,
    limit=None
):
    results = load_rc_checkpoint(
        checkpoint_path
    )

    successful_ids = {
        example_id
        for example_id, result in results.items()
        if result.get("status") == "success"
    }

    if limit is not None:
        work_df = dataset.iloc[:limit].copy()
    else:
        work_df = dataset.copy()

    work_ids = set(
        work_df["example_id"].astype(int)
    )

    already_done = len(
        successful_ids.intersection(work_ids)
    )

    total = len(work_df)

    print(f"Total examples: {total}")
    print(f"Already completed: {already_done}")
    print(f"Remaining: {total - already_done}")
    print("-" * 60)

    for _, row in work_df.iterrows():

        example_id = int(
            row["example_id"]
        )

        if example_id in successful_ids:
            continue

        prompt = build_rc_prompt(row)

        reference = str(
            row["output"]
        ).strip()

        try:
            torch.cuda.reset_peak_memory_stats()

            (
                completion,
                latency,
                output_tokens
            ) = generate_rc_answer(prompt)

            clean_prediction = clean_answer(
                completion
            )

            clean_reference = clean_answer(
                reference
            )

            f1 = token_f1(
                clean_prediction,
                clean_reference
            )

            peak_vram_gb = (
                torch.cuda.max_memory_allocated()
                / 1024**3
            )

            tokens_per_sec = (
                output_tokens / latency
                if latency > 0
                else None
            )

            results[example_id] = {
                "model_name": MODEL_NAME,
                "model_slug": MODEL_SLUG,

                "example_id": example_id,

                "instruction": row["instruction"],
                "input": row["input"],

                "reference": reference,
                "prediction": completion,

                "clean_reference": clean_reference,
                "clean_prediction": clean_prediction,

                "f1": f1,

                "latency_sec": latency,
                "output_tokens": output_tokens,
                "tokens_per_sec": tokens_per_sec,
                "peak_vram_gb": peak_vram_gb,

                "status": "success",
                "error": "",
            }

            successful_ids.add(
                example_id
            )

        except Exception as e:

            results[example_id] = {
                "model_name": MODEL_NAME,
                "model_slug": MODEL_SLUG,

                "example_id": example_id,

                "instruction": row["instruction"],
                "input": row["input"],

                "reference": reference,
                "prediction": "",

                "clean_reference":
                    clean_answer(reference),

                "clean_prediction": "",

                "f1": 0.0,

                "latency_sec": None,
                "output_tokens": None,
                "tokens_per_sec": None,
                "peak_vram_gb": None,

                "status": "error",
                "error": str(e),
            }

            print(
                f"\nERROR example {example_id}: {e}"
            )

        # Save after EVERY example
        save_rc_checkpoint(
            results,
            checkpoint_path
        )

        completed = len(
            successful_ids.intersection(
                work_ids
            )
        )

        print(
            f"\r{completed}/{total} completed | "
            f"ID {example_id} | "
            f"F1={results[example_id]['f1']:.3f}",
            end=""
        )

    print("\nRC evaluation finished.")

    return (
        pd.DataFrame(
            list(results.values())
        )
        .sort_values("example_id")
    )

In [122]:
rc_smoke_df = run_rc_evaluation(
    rc_df,
    RC_CHECKPOINT_PATH,
    limit=10
)

No RC checkpoint found.
Total examples: 10
Already completed: 0
Remaining: 10
------------------------------------------------------------
10/10 completed | ID 10 | F1=0.250
RC evaluation finished.


In [123]:
display(
    rc_smoke_df[
        [
            "example_id",
            "reference",
            "prediction",
            "f1",
            "latency_sec"
        ]
    ].head(10)
)

print(
    "Mean smoke F1:",
    rc_smoke_df[
        rc_smoke_df["example_id"] <= 10
    ]["f1"].mean()
)

,example_id,reference,prediction,f1,latency_sec
0,1,جواب: نزدیکی روابط لهستان به آمریکا,به دلیل روابط گرم بین لهستان و آمریکا و نزدیکی...,0.370370,4.640197
1,2,جواب: ایلیا سلیمان,ایلیا سلیمان,1.000000,0.765776
2,3,جواب: ۱۷ تن از امیران ارتش,عباس قره‌باغی فقط، آخرین ارتشبدی ارتش شاهنشاهی...,0.142857,2.920603
3,4,جواب: مشخصه بیماران مبتلا به اختلال شخصیت بدگم...,پارانوئید یک اختلال شخصیت است.,0.285714,1.448388
4,5,جواب: اعتقاد بر این است که سروتونین که یک انتق...,شخصانی که سطح بی‌روزی‌باز (BDNF) پایین است و د...,0.173913,5.850692
5,6,جواب: عفونت باکتریایی، انگلی یا ویروسی,عفونت باکتریایی یا ویروسی، ممکن است باعث اسهال...,0.571429,2.870420
6,7,جواب: طیف وسیعی از میکروارگانیسم‌های بیماری‌زا...,عواملی که باعث آلودگی آب می‌شوند عبارت‌اند از:...,0.304348,5.270551
7,8,جواب: اگر شما در حال تلاش برای چاق شدن هستید، ...,برای چاقی خوب، مصرف پروتئین بالاکاره با احتمال...,0.391304,4.766680
8,9,جواب: گوش‌,گوش‌ها تا آخر عمر انسان به رشد ادامه می‌یابند.,0.166667,1.910101
9,10,جواب: شماره‌های ایرانسل منطقه بندی ندارند,بر اساس متن، معرفی شماره ایرانسل به طور کلی نا...,0.250000,5.321763


Mean smoke F1: 0.36566022544283416


In [124]:
results_df = run_rc_evaluation(
    rc_df,
    RC_CHECKPOINT_PATH
)

Loaded RC checkpoint: 10 rows
Total examples: 600
Already completed: 10
Remaining: 590
------------------------------------------------------------
600/600 completed | ID 600 | F1=0.759
RC evaluation finished.


In [125]:
 results_df = pd.read_csv(
    RC_CHECKPOINT_PATH
)

successful_mask = (
    results_df["status"] == "success"
)

successful = results_df[
    successful_mask
].copy()

print(
    "Responses to score:",
    len(successful)
)

Responses to score: 600


In [126]:
predictions = (
    successful["clean_prediction"]
    .fillna("")
    .astype(str)
    .tolist()
)

references = (
    successful["clean_reference"]
    .fillna("")
    .astype(str)
    .tolist()
)

pred_embeddings = semantic_model.encode(
    predictions,
    batch_size=32,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

ref_embeddings = semantic_model.encode(
    references,
    batch_size=32,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

In [127]:
similarities = (
    torch.sum(
        pred_embeddings * ref_embeddings,
        dim=1
    )
    .cpu()
    .numpy()
)

successful[
    "semantic_similarity"
] = similarities

In [128]:
similarity_map = dict(
    zip(
        successful["example_id"],
        successful["semantic_similarity"]
    )
)

results_df[
    "semantic_similarity"
] = results_df[
    "example_id"
].map(
    similarity_map
)

In [129]:
results_df.to_csv(
    RC_SCORED_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Saved scored RC results:")
print(RC_SCORED_PATH)

Saved scored RC results:
/content/drive/MyDrive/llm_benchmark/parsbench/qwen2.5-3b-instruct_parsinlu_rc_scored.csv


In [130]:
successful = results_df[
    results_df["status"] == "success"
].copy()

rc_summary = pd.DataFrame([{
    "model_name": MODEL_NAME,
    "model_slug": MODEL_SLUG,

    "questions": len(successful),

    "mean_f1":
        successful["f1"].mean(),

    "median_f1":
        successful["f1"].median(),

    "mean_semantic_similarity":
        successful[
            "semantic_similarity"
        ].mean(),

    "median_semantic_similarity":
        successful[
            "semantic_similarity"
        ].median(),

    "avg_latency_sec":
        successful[
            "latency_sec"
        ].mean(),

    "avg_output_tokens":
        successful[
            "output_tokens"
        ].mean(),

    "avg_tokens_per_sec":
        successful[
            "tokens_per_sec"
        ].mean(),

    "peak_vram_gb":
        successful[
            "peak_vram_gb"
        ].max(),

    "errors":
        int(
            (results_df["status"] == "error")
            .sum()
        ),
}])

display(rc_summary)

,model_name,model_slug,questions,mean_f1,median_f1,mean_semantic_similarity,median_semantic_similarity,avg_latency_sec,avg_output_tokens,avg_tokens_per_sec,peak_vram_gb,errors
0,Qwen/Qwen2.5-3B-Instruct,qwen2.5-3b-instruct,600,0.364965,0.309402,0.664687,0.686543,2.546617,32.308333,12.256243,8.846513,0


In [131]:
results_df = pd.read_csv(
    RC_SCORED_PATH
)

def exact_match(prediction, reference):
    pred = normalize_fa_text(
        clean_answer(prediction)
    )

    ref = normalize_fa_text(
        clean_answer(reference)
    )

    return int(pred == ref)


results_df["exact_match"] = results_df.apply(
    lambda row: exact_match(
        row["prediction"],
        row["reference"]
    )
    if row["status"] == "success"
    else 0,
    axis=1
)

results_df.to_csv(
    RC_SCORED_PATH,
    index=False,
    encoding="utf-8-sig"
)

successful = results_df[
    results_df["status"] == "success"
].copy()

print(
    f"Exact Match: "
    f"{successful['exact_match'].mean():.4f}"
)

print(
    f"Mean F1: "
    f"{successful['f1'].mean():.4f}"
)

print(
    f"Mean Semantic Similarity: "
    f"{successful['semantic_similarity'].mean():.4f}"
)

Exact Match: 0.0633
Mean F1: 0.3650
Mean Semantic Similarity: 0.6647


In [132]:
rc_summary = pd.DataFrame([{
    "model_name": MODEL_NAME,
    "model_slug": MODEL_SLUG,

    "questions": len(successful),

    "exact_match":
        successful["exact_match"].mean(),

    "mean_f1":
        successful["f1"].mean(),

    "median_f1":
        successful["f1"].median(),

    "mean_semantic_similarity":
        successful[
            "semantic_similarity"
        ].mean(),

    "median_semantic_similarity":
        successful[
            "semantic_similarity"
        ].median(),

    "avg_latency_sec":
        successful[
            "latency_sec"
        ].mean(),

    "avg_output_tokens":
        successful[
            "output_tokens"
        ].mean(),

    "avg_tokens_per_sec":
        successful[
            "tokens_per_sec"
        ].mean(),

    "peak_vram_gb":
        successful[
            "peak_vram_gb"
        ].max(),

    "errors":
        int(
            (results_df["status"] == "error")
            .sum()
        ),
}])

rc_summary.to_csv(
    RC_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

display(rc_summary)

,model_name,model_slug,questions,exact_match,mean_f1,median_f1,mean_semantic_similarity,median_semantic_similarity,avg_latency_sec,avg_output_tokens,avg_tokens_per_sec,peak_vram_gb,errors
0,Qwen/Qwen2.5-3B-Instruct,qwen2.5-3b-instruct,600,0.063333,0.364965,0.309402,0.664687,0.686543,2.546617,32.308333,12.256243,8.846513,0


### Qwen 2.5 7B

In [102]:
RC_CHECKPOINT_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_rc_checkpoint.csv"
)

RC_SCORED_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_rc_scored.csv"
)

RC_SUMMARY_PATH = (
    BASE_DIR /
    f"{MODEL_SLUG}_parsinlu_rc_summary.csv"
)

print("Checkpoint:", RC_CHECKPOINT_PATH)
print("Scored results:", RC_SCORED_PATH)
print("Summary:", RC_SUMMARY_PATH)

Checkpoint: /content/drive/MyDrive/llm_benchmark/parsbench/qwen2.5-7b-instruct_parsinlu_rc_checkpoint.csv
Scored results: /content/drive/MyDrive/llm_benchmark/parsbench/qwen2.5-7b-instruct_parsinlu_rc_scored.csv
Summary: /content/drive/MyDrive/llm_benchmark/parsbench/qwen2.5-7b-instruct_parsinlu_rc_summary.csv


In [103]:
import os
import pandas as pd


def load_rc_checkpoint(path):
    if not path.exists():
        print("No RC checkpoint found.")
        return {}

    df = pd.read_csv(path)

    results = {
        int(row["example_id"]): row.to_dict()
        for _, row in df.iterrows()
    }

    print(f"Loaded RC checkpoint: {len(results)} rows")

    return results


def save_rc_checkpoint(results, path):
    if not results:
        return

    df = (
        pd.DataFrame(list(results.values()))
        .sort_values("example_id")
    )

    tmp_path = str(path) + ".tmp"

    df.to_csv(
        tmp_path,
        index=False,
        encoding="utf-8-sig"
    )

    os.replace(tmp_path, path)

In [104]:
def run_rc_evaluation(
    dataset,
    checkpoint_path,
    limit=None
):
    results = load_rc_checkpoint(
        checkpoint_path
    )

    successful_ids = {
        example_id
        for example_id, result in results.items()
        if result.get("status") == "success"
    }

    if limit is not None:
        work_df = dataset.iloc[:limit].copy()
    else:
        work_df = dataset.copy()

    work_ids = set(
        work_df["example_id"].astype(int)
    )

    already_done = len(
        successful_ids.intersection(work_ids)
    )

    total = len(work_df)

    print(f"Total examples: {total}")
    print(f"Already completed: {already_done}")
    print(f"Remaining: {total - already_done}")
    print("-" * 60)

    for _, row in work_df.iterrows():

        example_id = int(
            row["example_id"]
        )

        if example_id in successful_ids:
            continue

        prompt = build_rc_prompt(row)

        reference = str(
            row["output"]
        ).strip()

        try:
            torch.cuda.reset_peak_memory_stats()

            (
                completion,
                latency,
                output_tokens
            ) = generate_rc_answer(prompt)

            clean_prediction = clean_answer(
                completion
            )

            clean_reference = clean_answer(
                reference
            )

            f1 = token_f1(
                clean_prediction,
                clean_reference
            )

            peak_vram_gb = (
                torch.cuda.max_memory_allocated()
                / 1024**3
            )

            tokens_per_sec = (
                output_tokens / latency
                if latency > 0
                else None
            )

            results[example_id] = {
                "model_name": MODEL_NAME,
                "model_slug": MODEL_SLUG,

                "example_id": example_id,

                "instruction": row["instruction"],
                "input": row["input"],

                "reference": reference,
                "prediction": completion,

                "clean_reference": clean_reference,
                "clean_prediction": clean_prediction,

                "f1": f1,

                "latency_sec": latency,
                "output_tokens": output_tokens,
                "tokens_per_sec": tokens_per_sec,
                "peak_vram_gb": peak_vram_gb,

                "status": "success",
                "error": "",
            }

            successful_ids.add(
                example_id
            )

        except Exception as e:

            results[example_id] = {
                "model_name": MODEL_NAME,
                "model_slug": MODEL_SLUG,

                "example_id": example_id,

                "instruction": row["instruction"],
                "input": row["input"],

                "reference": reference,
                "prediction": "",

                "clean_reference":
                    clean_answer(reference),

                "clean_prediction": "",

                "f1": 0.0,

                "latency_sec": None,
                "output_tokens": None,
                "tokens_per_sec": None,
                "peak_vram_gb": None,

                "status": "error",
                "error": str(e),
            }

            print(
                f"\nERROR example {example_id}: {e}"
            )

        # Save after EVERY example
        save_rc_checkpoint(
            results,
            checkpoint_path
        )

        completed = len(
            successful_ids.intersection(
                work_ids
            )
        )

        print(
            f"\r{completed}/{total} completed | "
            f"ID {example_id} | "
            f"F1={results[example_id]['f1']:.3f}",
            end=""
        )

    print("\nRC evaluation finished.")

    return (
        pd.DataFrame(
            list(results.values())
        )
        .sort_values("example_id")
    )

In [105]:
rc_smoke_df = run_rc_evaluation(
    rc_df,
    RC_CHECKPOINT_PATH,
    limit=10
)

No RC checkpoint found.
Total examples: 10
Already completed: 0
Remaining: 10
------------------------------------------------------------
10/10 completed | ID 10 | F1=0.385
RC evaluation finished.


In [106]:
display(
    rc_smoke_df[
        [
            "example_id",
            "reference",
            "prediction",
            "f1",
            "latency_sec"
        ]
    ].head(10)
)

print(
    "Mean smoke F1:",
    rc_smoke_df[
        rc_smoke_df["example_id"] <= 10
    ]["f1"].mean()
)

,example_id,reference,prediction,f1,latency_sec
0,1,جواب: نزدیکی روابط لهستان به آمریکا,لهستان به دلیل وفیدگی خود به آمریکا و نزدیکی ر...,0.384615,4.702993
1,2,جواب: ایلیا سلیمان,ایلیا سلیمان,1.000000,1.484643
2,3,جواب: ۱۷ تن از امیران ارتش,در متن ذکر شده، ارتشبدی تنها برای امیران ارتش ...,0.153846,8.461773
3,4,جواب: مشخصه بیماران مبتلا به اختلال شخصیت بدگم...,پارانوئید یک نوع اختلال شخصیت است.,0.275862,2.937657
4,5,جواب: اعتقاد بر این است که سروتونین که یک انتق...,پاسخ دقیق این که کسانی خودکشی می‌کنند وجود ندا...,0.279070,5.683428
5,6,جواب: عفونت باکتریایی، انگلی یا ویروسی,عفونت باکتریایی، انگلی یا ویروسی باکتریها و ژی...,0.434783,4.196674
6,7,جواب: طیف وسیعی از میکروارگانیسم‌های بیماری‌زا...,آلودگی آب به دلیل وجود میکروارگانیسم‌های بیمار...,0.533333,4.801866
7,8,جواب: اگر شما در حال تلاش برای چاق شدن هستید، ...,پروتئین.,0.080000,1.448883
8,9,جواب: گوش‌,گوش تا آخر عمر ادامه دارد.,0.285714,1.451986
9,10,جواب: شماره‌های ایرانسل منطقه بندی ندارند,ایرانسل شماره‌های منطقه‌بندی ندارد. شماره‌های ...,0.384615,4.678883


Mean smoke F1: 0.3811838987227572


In [108]:
results_df = run_rc_evaluation(
    rc_df,
    RC_CHECKPOINT_PATH
)

Loaded RC checkpoint: 10 rows
Total examples: 600
Already completed: 10
Remaining: 590
------------------------------------------------------------
600/600 completed | ID 600 | F1=0.625
RC evaluation finished.


In [109]:
 results_df = pd.read_csv(
    RC_CHECKPOINT_PATH
)

successful_mask = (
    results_df["status"] == "success"
)

successful = results_df[
    successful_mask
].copy()

print(
    "Responses to score:",
    len(successful)
)

Responses to score: 600


In [110]:
predictions = (
    successful["clean_prediction"]
    .fillna("")
    .astype(str)
    .tolist()
)

references = (
    successful["clean_reference"]
    .fillna("")
    .astype(str)
    .tolist()
)

pred_embeddings = semantic_model.encode(
    predictions,
    batch_size=32,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

ref_embeddings = semantic_model.encode(
    references,
    batch_size=32,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

In [111]:
similarities = (
    torch.sum(
        pred_embeddings * ref_embeddings,
        dim=1
    )
    .cpu()
    .numpy()
)

successful[
    "semantic_similarity"
] = similarities

In [112]:
similarity_map = dict(
    zip(
        successful["example_id"],
        successful["semantic_similarity"]
    )
)

results_df[
    "semantic_similarity"
] = results_df[
    "example_id"
].map(
    similarity_map
)

In [113]:
results_df.to_csv(
    RC_SCORED_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Saved scored RC results:")
print(RC_SCORED_PATH)

Saved scored RC results:
/content/drive/MyDrive/llm_benchmark/parsbench/qwen2.5-7b-instruct_parsinlu_rc_scored.csv


In [114]:
successful = results_df[
    results_df["status"] == "success"
].copy()

rc_summary = pd.DataFrame([{
    "model_name": MODEL_NAME,
    "model_slug": MODEL_SLUG,

    "questions": len(successful),

    "mean_f1":
        successful["f1"].mean(),

    "median_f1":
        successful["f1"].median(),

    "mean_semantic_similarity":
        successful[
            "semantic_similarity"
        ].mean(),

    "median_semantic_similarity":
        successful[
            "semantic_similarity"
        ].median(),

    "avg_latency_sec":
        successful[
            "latency_sec"
        ].mean(),

    "avg_output_tokens":
        successful[
            "output_tokens"
        ].mean(),

    "avg_tokens_per_sec":
        successful[
            "tokens_per_sec"
        ].mean(),

    "peak_vram_gb":
        successful[
            "peak_vram_gb"
        ].max(),

    "errors":
        int(
            (results_df["status"] == "error")
            .sum()
        ),
}])

display(rc_summary)

,model_name,model_slug,questions,mean_f1,median_f1,mean_semantic_similarity,median_semantic_similarity,avg_latency_sec,avg_output_tokens,avg_tokens_per_sec,peak_vram_gb,errors
0,Qwen/Qwen2.5-7B-Instruct,qwen2.5-7b-instruct,600,0.440904,0.4,0.715972,0.74041,2.541777,32.376667,11.513585,12.50448,0


In [115]:
results_df = pd.read_csv(
    RC_SCORED_PATH
)

def exact_match(prediction, reference):
    pred = normalize_fa_text(
        clean_answer(prediction)
    )

    ref = normalize_fa_text(
        clean_answer(reference)
    )

    return int(pred == ref)


results_df["exact_match"] = results_df.apply(
    lambda row: exact_match(
        row["prediction"],
        row["reference"]
    )
    if row["status"] == "success"
    else 0,
    axis=1
)

results_df.to_csv(
    RC_SCORED_PATH,
    index=False,
    encoding="utf-8-sig"
)

successful = results_df[
    results_df["status"] == "success"
].copy()

print(
    f"Exact Match: "
    f"{successful['exact_match'].mean():.4f}"
)

print(
    f"Mean F1: "
    f"{successful['f1'].mean():.4f}"
)

print(
    f"Mean Semantic Similarity: "
    f"{successful['semantic_similarity'].mean():.4f}"
)

Exact Match: 0.1350
Mean F1: 0.4409
Mean Semantic Similarity: 0.7160


In [116]:
rc_summary = pd.DataFrame([{
    "model_name": MODEL_NAME,
    "model_slug": MODEL_SLUG,

    "questions": len(successful),

    "exact_match":
        successful["exact_match"].mean(),

    "mean_f1":
        successful["f1"].mean(),

    "median_f1":
        successful["f1"].median(),

    "mean_semantic_similarity":
        successful[
            "semantic_similarity"
        ].mean(),

    "median_semantic_similarity":
        successful[
            "semantic_similarity"
        ].median(),

    "avg_latency_sec":
        successful[
            "latency_sec"
        ].mean(),

    "avg_output_tokens":
        successful[
            "output_tokens"
        ].mean(),

    "avg_tokens_per_sec":
        successful[
            "tokens_per_sec"
        ].mean(),

    "peak_vram_gb":
        successful[
            "peak_vram_gb"
        ].max(),

    "errors":
        int(
            (results_df["status"] == "error")
            .sum()
        ),
}])

rc_summary.to_csv(
    RC_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

display(rc_summary)

,model_name,model_slug,questions,exact_match,mean_f1,median_f1,mean_semantic_similarity,median_semantic_similarity,avg_latency_sec,avg_output_tokens,avg_tokens_per_sec,peak_vram_gb,errors
0,Qwen/Qwen2.5-7B-Instruct,qwen2.5-7b-instruct,600,0.135,0.440904,0.4,0.715972,0.74041,2.541777,32.376667,11.513585,12.50448,0


### Load all models for this test

In [58]:
import gc
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)


def unload_model():
    global model, tokenizer

    try:
        del model
    except:
        pass

    try:
        del tokenizer
    except:
        pass

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    print("Previous model unloaded.")


def load_model(model_name):
    global model, tokenizer

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(
        model_name
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
    )

    model.eval()

    print("Loaded:", model_name)
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else "CPU"
    )

In [56]:
MODELS = {
    "dorna2-8b": "PartAI/Dorna2-Llama3.1-8B-Instruct",

    "gemma2-2b-it": "google/gemma-2-2b-it",

    "qwen2.5-3b-instruct": "Qwen/Qwen2.5-3B-Instruct",

    "qwen2.5-7b-instruct": "Qwen/Qwen2.5-7B-Instruct",
}

In [49]:
successful = results_df[
    results_df["status"] == "success"
].copy()

summary = (
    successful
    .groupby("category")
    .agg(
        questions=("example_id", "count"),
        correct=("correct", "sum"),
        accuracy=("correct", "mean"),
        avg_latency_sec=("latency_sec", "mean"),
        avg_tokens_per_sec=("tokens_per_sec", "mean"),
        peak_vram_gb=("peak_vram_gb", "max"),
    )
)

display(summary)

overall_accuracy = successful["correct"].mean()

print(
    f"Overall accuracy: {overall_accuracy:.2%}"
)

,questions,correct,accuracy,avg_latency_sec,avg_tokens_per_sec,peak_vram_gb
category,,,,,,
common_knowledge,350,143,0.408571,0.697720,9.719453,5.484757
literature,350,131,0.374286,0.695774,9.567498,5.509859
math_and_logic,350,114,0.325714,0.779778,10.257342,5.487520


Overall accuracy: 36.95%


In [35]:
smoke_df = run_mcq_evaluation(
    eval_df,
    CHECKPOINT_PATH,
    limit=30
)

No checkpoint found.
Total examples: 30
Already completed: 0
Remaining: 30
------------------------------------------------------------
30/30 completed | ID 30 | math_and_logic | target=1 | pred=2
Evaluation finished.


In [36]:
display(
    smoke_df[
        [
            "example_id",
            "category",
            "target",
            "raw_completion",
            "normalized_completion",
            "correct"
        ]
    ].head(30)
)

,example_id,category,target,raw_completion,normalized_completion,correct
0,1,math_and_logic,2,گزینه 2: 2A+B,2,1
1,2,math_and_logic,2,گزینه 2: ۴۱,2,1
2,3,math_and_logic,3,گزینه 2,2,0
3,4,math_and_logic,1,گزینه 2: 11,2,0
4,5,math_and_logic,1,گزینه 2,2,0
5,6,math_and_logic,4,جواب: 4,4,1
6,7,math_and_logic,2,جواب: ۲,2,1
7,8,math_and_logic,2,جواب: ۱,1,0
8,9,math_and_logic,3,گزینه صحیح: ۳. ۳۳.۳۳,3,1
9,10,math_and_logic,1,جواب: 2,2,0


In [37]:
smoke_success = smoke_df[
    (smoke_df["status"] == "success") &
    (smoke_df["example_id"] <= 30)
].copy()

print(
    "Smoke-test accuracy:",
    smoke_success["correct"].mean()
)

display(
    smoke_success.groupby("category")["correct"]
    .agg(["count", "sum", "mean"])
)

Smoke-test accuracy: 0.36666666666666664


,count,sum,mean
category,,,
math_and_logic,30,11,0.366667


In [38]:
smoke_eval_df = (
    eval_df
    .groupby(
        "category",
        group_keys=False
    )
    .head(10)
    .reset_index(drop=True)
)

print(
    smoke_eval_df["category"].value_counts()
)

category
math_and_logic      10
common_knowledge    10
literature          10
Name: count, dtype: int64


In [12]:
from pathlib import Path
from parsbench.tasks import ParsiNLUMultipleChoice

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/llm_benchmark/parsbench/dorna2_mcq_smoke"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with ParsiNLUMultipleChoice() as task:
    result = task.evaluate(
        dorna,
        prompt_lang="fa",
        prompt_shots=[0],
        n_first=10,
        save_matches=True,
        save_evaluation=True,
        output_path=str(OUTPUT_DIR),
    )

print(result)

Evaluating sub task 'math_and_logic' with 0-shot prompt:


Scoring matches: 100%|██████████| 10/10 [00:00<00:00, 94254.02it/s]


Evaluating sub task 'common_knowledge' with 0-shot prompt:


Scoring matches: 100%|██████████| 10/10 [00:00<00:00, 76538.39it/s]


Evaluating sub task 'literature' with 0-shot prompt:


Scoring matches: 100%|██████████| 10/10 [00:00<00:00, 87746.95it/s]

[EvaluationResult(model_name='PartAI/Dorna2-Llama3.1-8B-Instruct', task_name='ParsiNLU Multiple Choice', task_category=<TaskCategory.KNOWLEDGE: 'knowledge'>, score_name='Exact Match', prompt_shot_results=[PromptShotEvaluationResult(n_shots=0, score=0.1)], sub_task='math_and_logic'), EvaluationResult(model_name='PartAI/Dorna2-Llama3.1-8B-Instruct', task_name='ParsiNLU Multiple Choice', task_category=<TaskCategory.KNOWLEDGE: 'knowledge'>, score_name='Exact Match', prompt_shot_results=[PromptShotEvaluationResult(n_shots=0, score=0.0)], sub_task='common_knowledge'), EvaluationResult(model_name='PartAI/Dorna2-Llama3.1-8B-Instruct', task_name='ParsiNLU Multiple Choice', task_category=<TaskCategory.KNOWLEDGE: 'knowledge'>, score_name='Exact Match', prompt_shot_results=[PromptShotEvaluationResult(n_shots=0, score=0.2)], sub_task='literature')]


In [15]:
import re

PERSIAN_DIGITS = str.maketrans({
    "۰": "0",
    "۱": "1",
    "۲": "2",
    "۳": "3",
    "۴": "4",
    "۵": "5",
    "۶": "6",
    "۷": "7",
    "۸": "8",
    "۹": "9",
    "٠": "0",
    "١": "1",
    "٢": "2",
    "٣": "3",
    "٤": "4",
    "٥": "5",
    "٦": "6",
    "٧": "7",
    "٨": "8",
    "٩": "9",
})


def normalize_mcq_answer(text):
    if text is None:
        return "-1"

    text = str(text).translate(PERSIAN_DIGITS).strip()

    # Exact clean answer
    if text in {"1", "2", "3", "4"}:
        return text

    # Find candidate option numbers
    matches = re.findall(r"(?<!\d)([1-4])(?!\d)", text)

    # Accept only if the response identifies one unique option
    unique = list(dict.fromkeys(matches))

    if len(unique) == 1:
        return unique[0]

    return "-1"

In [16]:
tests = [
    "2",
    "۲",
    "گزینه 4: A-B",
    "جواب صحیح: **2**",
    "جواب: 4. باران است با برف",
]

for x in tests:
    print(x, "->", normalize_mcq_answer(x))

2 -> 2
۲ -> 2
گزینه 4: A-B -> 4
جواب صحیح: **2** -> 2
جواب: 4. باران است با برف -> 4


In [13]:
!find "/content/drive/MyDrive/llm_benchmark/parsbench/dorna2_mcq_smoke" \
    -type f -maxdepth 3 -print

find: warning: you have specified the global option -maxdepth after the argument -type, but global options are not positional, i.e., -maxdepth affects tests specified before it as well as those specified after it.  Please specify global options before other arguments.
/content/drive/MyDrive/llm_benchmark/parsbench/dorna2_mcq_smoke/PartAI_Dorna2_Llama3.1_8B_Instruct/ParsiNLU_Multiple_Choice/matches_common_knowledge_0_shot.jsonl
/content/drive/MyDrive/llm_benchmark/parsbench/dorna2_mcq_smoke/PartAI_Dorna2_Llama3.1_8B_Instruct/ParsiNLU_Multiple_Choice/evaluation_literature.jsonl
/content/drive/MyDrive/llm_benchmark/parsbench/dorna2_mcq_smoke/PartAI_Dorna2_Llama3.1_8B_Instruct/ParsiNLU_Multiple_Choice/evaluation_math_and_logic.jsonl
/content/drive/MyDrive/llm_benchmark/parsbench/dorna2_mcq_smoke/PartAI_Dorna2_Llama3.1_8B_Instruct/ParsiNLU_Multiple_Choice/matches_math_and_logic_0_shot.jsonl
/content/drive/MyDrive/llm_benchmark/parsbench/dorna2_mcq_smoke/PartAI_Dorna2_Llama3.1_8B_Instruct/Pa

In [14]:
import os
import json

for root, dirs, files in os.walk(OUTPUT_DIR):
    for file in files:
        path = os.path.join(root, file)
        print("\nFILE:", path)

        if file.endswith((".json", ".jsonl")):
            with open(path, "r", encoding="utf-8") as f:
                for i, line in enumerate(f):
                    print(line[:2000])
                    if i >= 2:
                        break


FILE: /content/drive/MyDrive/llm_benchmark/parsbench/dorna2_mcq_smoke/PartAI_Dorna2_Llama3.1_8B_Instruct/ParsiNLU_Multiple_Choice/matches_common_knowledge_0_shot.jsonl
{"id": 1, "prompt": "در ادامه، به شما یک سوال چند گزینه‌ای به زبان فارسی نشان داده می شود. شما باید بر اساس دانش خود به سوال پاسخ دهید. پاسخ خود را از بین گزینه‌های داده شده انتخاب کنید.\nفقط عدد متناظر با گزینه درست را خروجی بده.\n\nسوال:\n'''کدام کشور اولین تولید کننده خرما در جهان است؟'''\nگزینه ها:\n'''1. ایران\n2. عربستان\n3. عراق\n4. سوریه'''\nجواب:", "target": "1", "completion": "2", "formatted_completion": "2", "score": 0}

{"id": 2, "prompt": "در ادامه، به شما یک سوال چند گزینه‌ای به زبان فارسی نشان داده می شود. شما باید بر اساس دانش خود به سوال پاسخ دهید. پاسخ خود را از بین گزینه‌های داده شده انتخاب کنید.\nفقط عدد متناظر با گزینه درست را خروجی بده.\n\nسوال:\n'''مقام رهبری در قانون اساسی جمهوری اسلامی دارای چه کار ویژه ای است؟'''\nگزینه ها:\n'''1. ریاست کشور\n2. نظارت عالیه\n3. تنظیم کننده قوای سه گانه\n4. حاکم

In [11]:
print(result)

[EvaluationResult(model_name='PartAI/Dorna2-Llama3.1-8B-Instruct', task_name='ParsiNLU Multiple Choice', task_category=<TaskCategory.KNOWLEDGE: 'knowledge'>, score_name='Exact Match', prompt_shot_results=[PromptShotEvaluationResult(n_shots=0, score=0.0)], sub_task='math_and_logic'), EvaluationResult(model_name='PartAI/Dorna2-Llama3.1-8B-Instruct', task_name='ParsiNLU Multiple Choice', task_category=<TaskCategory.KNOWLEDGE: 'knowledge'>, score_name='Exact Match', prompt_shot_results=[PromptShotEvaluationResult(n_shots=0, score=0.0)], sub_task='common_knowledge'), EvaluationResult(model_name='PartAI/Dorna2-Llama3.1-8B-Instruct', task_name='ParsiNLU Multiple Choice', task_category=<TaskCategory.KNOWLEDGE: 'knowledge'>, score_name='Exact Match', prompt_shot_results=[PromptShotEvaluationResult(n_shots=0, score=0.0)], sub_task='literature')]


In [7]:
!python --version

Python 3.12.13


In [ ]:
with ParsiNLUMultipleChoice() as task:
    mcq_result = task.evaluate(
        dorna,
        prompt_lang="fa",
        prompt_shots=0
    )

Prevously

In [ ]:
!pip -q install bert-score evaluate rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00


In [ ]:
!pip install -U transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.9.0
    Uninstalling transformers-5.9.0:
      Successfully uninstalled transformers-5.9.0


In [ ]:
import pandas as pd

eval_df = pd.read_csv(
    "/content/drive/MyDrive/llm_benchmark/persian_chatbot_eval_1000.csv"
)

pilot_df = eval_df.iloc[:1000].copy()

print(pilot_df.shape)
pilot_df.head()

(1000, 3)


,sample_id,inputs,outputs
0,0,در این مسئله متن را کامل بخوان و بهترین جواب ر...,شهر گواتمالاسیتی
1,1,در این مسئله متن را کامل بخوان و بهترین جواب ر...,زوال عقل سالخوردگی یا بیماری آلزایمر
2,2,در این مسئله متن را کامل بخوان و بهترین جواب ر...,جرماغون نویان
3,3,در این مسئله متن را کامل بخوان و بهترین جواب ر...,برای نخستین بار در ساخت آن، هم از قطعات مکانیک...
4,4,در این مسئله متن را کامل بخوان و بهترین جواب ر...,ازدواج دخترش تاج الملوک با مظفرالدین شاه قاجار...


In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_NAME = "PartAI/Dorna2-Llama3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16
)

print("Loaded successfully")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.3k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Loaded successfully


In [ ]:
print("Loaded:", MODEL_NAME)

Loaded: PartAI/Dorna2-Llama3.1-8B-Instruct


In [ ]:
def generate_answer(question):

    messages = [
        {
            "role": "system",
            "content": """
شما یک سامانه پرسش و پاسخ فارسی هستید.

قوانین:
- فقط پاسخ نهایی را بنویس.
- توضیح نده.
- استدلال نکن.
- مقدمه ننویس.
- فقط خود پاسخ را برگردان.
"""
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=32,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return generated.strip()

In [ ]:
for q in [
    "پایتخت ایران چیست؟",
    "بزرگترین سیاره منظومه شمسی چیست؟",
    "معروف ترین نوع زوال عقل چیست؟"
]:
    print("Q:", q)
    print("A:", generate_answer(q))
    print("-"*50)

Q: پایتخت ایران چیست؟
A: تهران
--------------------------------------------------
Q: بزرگترین سیاره منظومه شمسی چیست؟
A: ژوپیتر
--------------------------------------------------
Q: معروف ترین نوع زوال عقل چیست؟
A: زوال عقل نوعی بیماری عصبی است که بر توانایی‌های شناختی و رفتاری فرد تاثیر می‌گذ
--------------------------------------------------


In [ ]:
print(model.quantization_method if hasattr(model, "quantization_method") else "No quantization_method")

QuantizationMethod.BITS_AND_BYTES


In [ ]:
for name, module in model.named_modules():
    if "Linear4bit" in str(type(module)):
        print("Found 4-bit layer:", name)
        break

Found 4-bit layer: model.layers.0.self_attn.q_proj


In [ ]:
print(model.config)

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "float16",
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "quantization_config": {
    "_load_in_4bit": true,
    "_load_in_8bit": false,
    "bnb_4bit_compute_dtype": "float16",
    "bnb_4bit_quant_storage": "uint8",
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": true,
    "llm_int8_enable_fp32_cpu_offload": false,
    "llm_int8_has_fp16_weight": false,
    "llm_int8_skip_modules": null,
    "llm_int8_threshold": 6.0,
    "load_in_4bit": true,
    "load_in_8bit": fa

In [ ]:
import time
import torch
import pandas as pd

predictions = []
references = []

latencies = []
token_counts = []

torch.cuda.reset_peak_memory_stats()

for idx, row in pilot_df.iterrows():

    prompt = row["inputs"]
    reference = row["outputs"]

    start = time.time()

    prediction = generate_answer(prompt)

    latency = time.time() - start

    pred_tokens = len(
        tokenizer.encode(
            prediction,
            add_special_tokens=False
        )
    )

    predictions.append(prediction)
    references.append(reference)

    latencies.append(latency)
    token_counts.append(pred_tokens)

    if (idx + 1) % 50 == 0:

        checkpoint_df = pd.DataFrame({
            "reference": references,
            "prediction": predictions,
            "latency": latencies,
            "tokens": token_counts
        })

        checkpoint_df.to_csv(
            "/content/drive/MyDrive/llm_benchmark/dorna2_partial.csv",
            index=False,
            encoding="utf-8-sig"
        )

        print(f"Checkpoint saved: {idx+1}")

    if (idx + 1) % 10 == 0:
        print(f"Completed {idx+1}/1000")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Completed 10/1000
Completed 20/1000
Completed 30/1000
Completed 40/1000
Checkpoint saved: 50
Completed 50/1000
Completed 60/1000
Completed 70/1000
Completed 80/1000
Completed 90/1000
Checkpoint saved: 100
Completed 100/1000
Completed 110/1000
Completed 120/1000
Completed 130/1000
Completed 140/1000
Checkpoint saved: 150
Completed 150/1000
Completed 160/1000
Completed 170/1000
Completed 180/1000
Completed 190/1000
Checkpoint saved: 200
Completed 200/1000
Completed 210/1000
Completed 220/1000
Completed 230/1000
Completed 240/1000
Checkpoint saved: 250
Completed 250/1000
Completed 260/1000
Completed 270/1000
Completed 280/1000
Completed 290/1000
Checkpoint saved: 300
Completed 300/1000
Completed 310/1000
Completed 320/1000
Completed 330/1000
Completed 340/1000
Checkpoint saved: 350
Completed 350/1000
Completed 360/1000
Completed 370/1000
Completed 380/1000
Completed 390/1000
Checkpoint saved: 400
Completed 400/1000
Completed 410/1000
Completed 420/1000
Completed 430/1000
Completed 440/100

In [ ]:
results_df = pd.DataFrame({
    "reference": references,
    "prediction": predictions,
    "latency": latencies,
    "tokens": token_counts
})

results_df.to_csv(
    "/content/drive/MyDrive/llm_benchmark/dorna2_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Results saved.")

Results saved.


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

sim_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)

pred_emb = sim_model.encode(
    predictions,
    batch_size=32,
    show_progress_bar=True
)

ref_emb = sim_model.encode(
    references,
    batch_size=32,
    show_progress_bar=True
)

similarities = [
    cosine_similarity(
        pred.reshape(1, -1),
        ref.reshape(1, -1)
    )[0][0]
    for pred, ref in zip(pred_emb, ref_emb)
]

semantic_similarity = float(np.mean(similarities))

print("Semantic Similarity:", semantic_similarity)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Semantic Similarity: 0.7292196154594421


In [ ]:
import evaluate

rouge = evaluate.load("rouge")

rouge_result = rouge.compute(
    predictions=predictions,
    references=references
)

rouge_l = rouge_result["rougeL"]

print("ROUGE-L:", rouge_l)

ROUGE-L: 0.010833333333333332


In [ ]:
avg_latency = sum(latencies) / len(latencies)

total_tokens = sum(token_counts)

total_time = sum(latencies)

tokens_per_sec = total_tokens / total_time

peak_vram = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

print("Average Latency:", avg_latency)
print("Tokens/sec:", tokens_per_sec)
print("Peak VRAM:", peak_vram)

Average Latency: 2.520556212425232
Tokens/sec: 8.552477383259093
Peak VRAM: 6.474133491516113


In [ ]:
summary_df = pd.DataFrame({
    "model": ["Dorna2-Llama3.1-8B-Instruct"],
    "num_samples": [len(predictions)],
    "semantic_similarity": [semantic_similarity],
    "rouge_l": [rouge_l],
    "avg_latency_sec": [avg_latency],
    "tokens_per_sec": [tokens_per_sec],
    "peak_vram_gb": [peak_vram]
})

summary_df.to_csv(
    "/content/drive/MyDrive/llm_benchmark/dorna2_summary.csv",
    index=False
)

summary_df

,model,num_samples,semantic_similarity,rouge_l,avg_latency_sec,tokens_per_sec,peak_vram_gb
0,Dorna2-Llama3.1-8B-Instruct,1000,0.72922,0.010833,2.520556,8.552477,6.474133
